## Setup and dataset paths

In [2]:

from pathlib import Path
import pandas as pd
import numpy as np
import random
import json

from rdkit import Chem, rdBase

# Reproducibility
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

# Molecular constraints
MAX_ATOMS = 100

# Notebook is inside REWORKED CODE/PREPROCESSING/
# Datasets are inside REWORKED CODE/DATA/

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent

DATA_ROOT = PROJECT_DIR / "DATA"

CHEMBL_DIR = DATA_ROOT / "chembl"
BINDINGDB_DIR = DATA_ROOT / "bindingdb"

CHEMBL_CSV = CHEMBL_DIR / "chembl.csv"
CHEMBL_TXT = CHEMBL_DIR / "chembl.txt"

TRAIN_DIR = BINDINGDB_DIR / "train_dataset"
TEST_DIR = BINDINGDB_DIR / "test_dataset"

# Verify locations
assert CHEMBL_CSV.is_file(), f"Missing: {CHEMBL_CSV}"
assert TRAIN_DIR.is_dir(), f"Missing: {TRAIN_DIR}"
assert TEST_DIR.is_dir(), f"Missing: {TEST_DIR}"

# Find BindingDB files across all folds
train_ligand_files = sorted(
    TRAIN_DIR.rglob("ligands_*_corrected")
)
test_ligand_files = sorted(
    TEST_DIR.rglob("ligands_*_corrected")
)

train_protein_files = sorted(
    TRAIN_DIR.rglob("proteins_*_corrected")
)
test_protein_files = sorted(
    TEST_DIR.rglob("proteins_*_corrected")
)

print("RDKit version:", rdBase.rdkitVersion)
print("Notebook directory:", NOTEBOOK_DIR)
print("Data directory:", DATA_ROOT)

print("\nChEMBL:")
print("CSV exists:", CHEMBL_CSV.exists())
print("TXT exists:", CHEMBL_TXT.exists())

print("\nBindingDB:")
print("Training ligand files:", len(train_ligand_files))
print("Testing ligand files:", len(test_ligand_files))
print("Training protein files:", len(train_protein_files))
print("Testing protein files:", len(test_protein_files))

print("\nTraining folds:")
for file in train_ligand_files:
    print(" ", file.parent.name)

print("\nTesting folds:")
for file in test_ligand_files:
    print(" ", file.parent.name)


RDKit version: 2026.03.6
Notebook directory: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING
Data directory: /home/muhamed/repo/CTDDG/REWORKED_CODE/DATA

ChEMBL:
CSV exists: True
TXT exists: True

BindingDB:
Training ligand files: 5
Testing ligand files: 5
Training protein files: 5
Testing protein files: 5

Training folds:
  train_4_org_1000_112
  train_4_org_1002_103
  train_4_org_1004_122
  train_4_org_1036_124
  train_4_org_1042_104

Testing folds:
  test_4_org_1000_112
  test_4_org_1002_103
  test_4_org_1004_122
  test_4_org_1036_124
  test_4_org_1042_104


## OUTPUT DIRECTORY

In [3]:

# Output directory alongside DATA and PREPROCESSING
OUTPUT_DIR = PROJECT_DIR / "PREPROCESSING_OUTPUT"

CHEMBL_OUTPUT = OUTPUT_DIR / "chembl"
BINDINGDB_OUTPUT = OUTPUT_DIR / "bindingdb"
EMBEDDINGS_OUTPUT = BINDINGDB_OUTPUT / "protein_embeddings"

for directory in [
    CHEMBL_OUTPUT,
    BINDINGDB_OUTPUT,
    EMBEDDINGS_OUTPUT
]:
    directory.mkdir(parents=True, exist_ok=True)

print("All outputs will be saved under:")
print(OUTPUT_DIR)


All outputs will be saved under:
/home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT


## Raw dataset inspection

In [4]:

import csv
from itertools import zip_longest

# ============================================================
# PART A: ChEMBL inspection
# ============================================================

print("=" * 65)
print("CHEMBL DATASET INSPECTION")
print("=" * 65)

with CHEMBL_CSV.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter=";")

    columns = reader.fieldnames
    print("\nCSV columns:", columns)

    if "Smiles" not in columns:
        raise ValueError("ChEMBL CSV has no 'Smiles' column!")

    chembl_count = 0
    missing_smiles = 0
    csv_examples = []

    for row in reader:
        chembl_count += 1

        smiles = (row["Smiles"] or "").strip()

        if not smiles:
            missing_smiles += 1
        elif len(csv_examples) < 3:
            csv_examples.append(smiles)

print("\nTotal CSV records:", chembl_count)
print("Missing SMILES:", missing_smiles)

print("\nSample ChEMBL SMILES:")
for s in csv_examples:
    print(" ", s)

if CHEMBL_TXT.exists():
    txt_count = 0
    txt_examples = []

    with CHEMBL_TXT.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                txt_count += 1
                if len(txt_examples) < 3:
                    txt_examples.append(line.strip())

    print("\nChEMBL TXT non-empty lines:", txt_count)
    print("TXT examples:")
    for s in txt_examples:
        print(" ", s)


# ============================================================
# PART B: BindingDB inspection
# ============================================================

print("\n" + "=" * 65)
print("BINDINGDB DATASET INSPECTION")
print("=" * 65)

fold_results = []

for split, ligand_files, protein_files in [
    ("Train", train_ligand_files, train_protein_files),
    ("Test", test_ligand_files, test_protein_files),
]:

    ligand_map = {p.parent.name: p for p in ligand_files}
    protein_map = {p.parent.name: p for p in protein_files}

    if set(ligand_map) != set(protein_map):
        raise ValueError(
            f"{split}: Ligand and protein fold folders differ!"
        )

    for fold in sorted(ligand_map):
        ligand_path = ligand_map[fold]
        protein_path = protein_map[fold]

        ligand_count = 0
        protein_count = 0
        blank_ligands = 0
        blank_proteins = 0

        ligand_example = None
        protein_example = None

        with (
            ligand_path.open("r", encoding="utf-8") as lf,
            protein_path.open("r", encoding="utf-8") as pf,
        ):
            for ligand, protein in zip_longest(lf, pf):

                if ligand is not None:
                    ligand_count += 1

                    if not ligand.strip():
                        blank_ligands += 1
                    elif ligand_example is None:
                        ligand_example = ligand.strip()

                if protein is not None:
                    protein_count += 1

                    if not protein.strip():
                        blank_proteins += 1
                    elif protein_example is None:
                        protein_example = protein.strip()

        aligned = (
            ligand_count == protein_count
            and blank_ligands == 0
            and blank_proteins == 0
        )

        fold_results.append({
            "split": split,
            "fold": fold,
            "ligands": ligand_count,
            "proteins": protein_count,
            "blank_ligands": blank_ligands,
            "blank_proteins": blank_proteins,
            "aligned": aligned,
        })

        print(f"\n{split} | {fold}")
        print("Ligands:", ligand_count)
        print("Proteins:", protein_count)
        print("Blank ligand lines:", blank_ligands)
        print("Blank protein lines:", blank_proteins)
        print("Line counts aligned:", aligned)

        if ligand_example:
            print("\nRaw ligand:")
            print(ligand_example[:120])

            print("\nReconstructed ligand SMILES:")
            print("".join(ligand_example.split())[:120])

        if protein_example:
            print("\nReconstructed protein sequence:")
            print("".join(protein_example.split())[:100])

        print("-" * 65)


# ============================================================
# PART C: Summary
# ============================================================

print("\n" + "=" * 65)
print("FINAL DATASET SUMMARY")
print("=" * 65)

for split in ["Train", "Test"]:
    records = [
        r for r in fold_results
        if r["split"] == split
    ]

    print(f"\n{split}:")
    print("Total ligand records:",
          sum(r["ligands"] for r in records))
    print("Total protein records:",
          sum(r["proteins"] for r in records))

print("\nAll folds aligned:",
      all(r["aligned"] for r in fold_results))


CHEMBL DATASET INSPECTION

CSV columns: ['ChEMBL ID', 'Name', 'Synonyms', 'Type', 'Max Phase', 'Molecular Weight', 'Targets', 'Bioactivities', 'AlogP', 'Polar Surface Area', 'HBA', 'HBD', '#RO5 Violations', '#Rotatable Bonds', 'Passes Ro3', 'QED Weighted', 'CX Acidic pKa', 'CX Basic pKa', 'CX LogP', 'CX LogD', 'Aromatic Rings', 'Structure Type', 'Inorganic Flag', 'Heavy Atoms', 'HBA (Lipinski)', 'HBD (Lipinski)', '#RO5 Violations (Lipinski)', 'Molecular Weight (Monoisotopic)', 'Molecular Species', 'Molecular Formula', 'Smiles']

Total CSV records: 1207360
Missing SMILES: 0

Sample ChEMBL SMILES:
  Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1
  O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1
  CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4)CC3)c12

ChEMBL TXT non-empty lines: 1090529
TXT examples:
  c1ccc(-n2cc(CNC3CCc4ncnn4C3)cn2)cc1
  Cc1cccc(CCN2CCC[C@@H]2C)c1
  CN(C)c1ccc(-c2cnc3c(N)ncnc3n2)cc1

BINDINGDB DATASET INSPECTION

Train | train_4_org_1000_112
Ligands: 143999
Proteins: 143999
Blank lig

## BindingDB Canonicalization and Overlap Index Creation

In [5]:

from collections import Counter
from functools import lru_cache
from itertools import zip_longest
from rdkit import Chem

# ============================================================
# BLOCK 3: BINDINGDB CANONICALIZATION
# ============================================================

@lru_cache(maxsize=200_000)
def standardize_smiles(smiles):
    """
    Parse SMILES and retain the largest heavy-atom fragment.
    Return (canonical_smiles, had_multiple_fragments).

    Preserve stereochemistry and formal charges.
    """
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None, False

    fragments = Chem.GetMolFrags(
        mol, asMols=True, sanitizeFrags=True
    )

    if not fragments:
        return None, False

    # Deterministic largest-fragment selection
    fragments = sorted(
        fragments,
        key=lambda m: (
            -m.GetNumHeavyAtoms(),
            -m.GetNumAtoms(),
            Chem.MolToSmiles(
                m, canonical=True, isomericSmiles=True
            )
        )
    )

    main_mol = fragments[0]

    canonical = Chem.MolToSmiles(
        main_mol,
        canonical=True,
        isomericSmiles=True
    )

    return canonical, len(fragments) > 1


bindingdb_exclusion_smiles = set()
bindingdb_fold_stats = []

print("=" * 65)
print("BINDINGDB CANONICALIZATION")
print("=" * 65)

for split, ligand_files, protein_files in [
    ("train", train_ligand_files, train_protein_files),
    ("test", test_ligand_files, test_protein_files),
]:

    protein_map = {
        p.parent.name: p for p in protein_files
    }

    for ligand_path in ligand_files:

        fold = ligand_path.parent.name
        protein_path = protein_map[fold]

        stats = Counter()
        unique_proteins = set()
        fold_smiles = set()

        with (
            ligand_path.open("r", encoding="utf-8") as lf,
            protein_path.open("r", encoding="utf-8") as pf,
        ):
            for ligand_line, protein_line in zip_longest(lf, pf):

                if ligand_line is None or protein_line is None:
                    raise ValueError(
                        f"Protein-ligand line mismatch: {fold}"
                    )

                stats["total_pairs"] += 1

                raw_smiles = "".join(ligand_line.split())
                protein = "".join(protein_line.split())

                if not raw_smiles or not protein:
                    raise ValueError(
                        f"Empty record in {fold}, "
                        f"line {stats['total_pairs']}"
                    )

                unique_proteins.add(protein)

                canonical, fragmented = standardize_smiles(
                    raw_smiles
                )

                if canonical is None:
                    stats["invalid_smiles"] += 1
                    continue

                stats["valid_smiles"] += 1

                if fragmented:
                    stats["fragmented_smiles"] += 1

                fold_smiles.add(canonical)
                bindingdb_exclusion_smiles.add(canonical)

        result = {
            "split": split,
            "fold": fold,
            "total_pairs": stats["total_pairs"],
            "valid_smiles": stats["valid_smiles"],
            "invalid_smiles": stats["invalid_smiles"],
            "fragmented_smiles": stats["fragmented_smiles"],
            "unique_ligands": len(fold_smiles),
            "unique_proteins": len(unique_proteins),
        }

        bindingdb_fold_stats.append(result)

        print(f"\n{split.upper()} | {fold}")
        for key, value in result.items():
            if key not in ("split", "fold"):
                print(f"  {key}: {value:,}")

print("\n" + "=" * 65)
print("BINDINGDB SUMMARY")
print("=" * 65)

print(
    "Total protein-ligand pairs:",
    f"{sum(r['total_pairs'] for r in bindingdb_fold_stats):,}"
)

print(
    "Total invalid ligand records:",
    f"{sum(r['invalid_smiles'] for r in bindingdb_fold_stats):,}"
)

print(
    "Unique standardized BindingDB molecules:",
    f"{len(bindingdb_exclusion_smiles):,}"
)

assert bindingdb_exclusion_smiles, (
    "No valid BindingDB molecules extracted!"
)

print("\nBindingDB overlap index created successfully.")


BINDINGDB CANONICALIZATION



TRAIN | train_4_org_1000_112
  total_pairs: 143,999
  valid_smiles: 143,999
  invalid_smiles: 0
  fragmented_smiles: 570
  unique_ligands: 98,622
  unique_proteins: 1,000

TRAIN | train_4_org_1002_103
  total_pairs: 144,885
  valid_smiles: 144,885
  invalid_smiles: 0
  fragmented_smiles: 558
  unique_ligands: 99,376
  unique_proteins: 1,002

TRAIN | train_4_org_1004_122
  total_pairs: 146,059
  valid_smiles: 146,059
  invalid_smiles: 0
  fragmented_smiles: 571
  unique_ligands: 100,237
  unique_proteins: 1,004

TRAIN | train_4_org_1036_124
  total_pairs: 142,687
  valid_smiles: 142,687
  invalid_smiles: 0
  fragmented_smiles: 566
  unique_ligands: 97,312
  unique_proteins: 1,036

TRAIN | train_4_org_1042_104
  total_pairs: 227,093
  valid_smiles: 227,093
  invalid_smiles: 0
  fragmented_smiles: 582
  unique_ligands: 147,681
  unique_proteins: 1,514

TEST | test_4_org_1000_112
  total_pairs: 16,200
  valid_smiles: 16,200
  invalid_smiles: 0
  fragmented_smiles: 18
  unique_ligands: 9,4

## Clean and standardize ChEMBL

In [6]:

import csv
from collections import Counter
from rdkit import Chem

# ============================================================
# BLOCK 4: CHEMBL PREPROCESSING
# ============================================================

# Use the standardization function from Block 3
assert callable(standardize_smiles), (
    "Run Block 3 before running Block 4."
)

# Save intermediate results outside the original DATA folder
CHEMBL_OUTPUT.mkdir(parents=True, exist_ok=True)

CHEMBL_INTERMEDIATE = (
    CHEMBL_OUTPUT / "chembl_standardized_pre_overlap.csv"
)

chembl_stats = Counter()
error_examples = []
sample_molecules = []

print("=" * 65)
print("CHEMBL MOLECULAR PREPROCESSING")
print("=" * 65)

with (
    CHEMBL_CSV.open(
        "r", encoding="utf-8-sig", newline=""
    ) as input_file,
    CHEMBL_INTERMEDIATE.open(
        "w", encoding="utf-8", newline=""
    ) as output_file,
):

    reader = csv.DictReader(input_file, delimiter=";")

    writer = csv.DictWriter(
        output_file,
        fieldnames=[
            "chembl_id",
            "original_smiles",
            "canonical_smiles",
            "num_atoms",
            "num_bonds",
            "fragmented",
        ],
    )

    writer.writeheader()

    for row in reader:

        chembl_stats["total"] += 1

        chembl_id = (row.get("ChEMBL ID") or "").strip()
        raw_smiles = (row.get("Smiles") or "").strip()

        # 1. Remove missing SMILES
        if not raw_smiles:
            chembl_stats["missing_smiles"] += 1
            continue

        # 2. Parse, fragment-select and canonicalize
        try:
            canonical, fragmented = standardize_smiles(
                raw_smiles
            )

            if canonical is None:
                chembl_stats["invalid_smiles"] += 1
                continue

            mol = Chem.MolFromSmiles(canonical)

            if mol is None:
                chembl_stats["invalid_smiles"] += 1
                continue

        except Exception as exc:
            chembl_stats["processing_errors"] += 1

            if len(error_examples) < 5:
                error_examples.append(
                    (chembl_id, str(exc))
                )
            continue

        if fragmented:
            chembl_stats["fragmented"] += 1

        # 3. Molecular graph statistics
        num_atoms = mol.GetNumAtoms()
        num_bonds = mol.GetNumBonds()

        # 4. Maximum atom limit
        if num_atoms > MAX_ATOMS:
            chembl_stats["too_many_atoms"] += 1
            continue

        # 5. Reject single-atom / zero-bond structures
        if num_bonds == 0:
            chembl_stats["zero_bonds"] += 1
            continue

        # 6. Save cleaned molecule
        writer.writerow({
            "chembl_id": chembl_id,
            "original_smiles": raw_smiles,
            "canonical_smiles": canonical,
            "num_atoms": num_atoms,
            "num_bonds": num_bonds,
            "fragmented": fragmented,
        })

        chembl_stats["retained"] += 1

        if len(sample_molecules) < 5:
            sample_molecules.append(
                (raw_smiles, canonical)
            )


# ============================================================
# SUMMARY
# ============================================================

print("\nCHEMBL PREPROCESSING SUMMARY")
print("-" * 65)

for key, value in chembl_stats.items():
    print(f"{key:25s}: {value:,}")

print("\nRetention rate: {:.2f}%".format(
    100 * chembl_stats["retained"] /
    max(chembl_stats["total"], 1)
))

print("\nSample standardized molecules:")

for original, canonical in sample_molecules:
    print("\nOriginal :", original)
    print("Canonical:", canonical)

if error_examples:
    print("\nProcessing error examples:")
    for example in error_examples:
        print(example)

# Verify that all input records were accounted for
excluded = (
    chembl_stats["missing_smiles"]
    + chembl_stats["invalid_smiles"]
    + chembl_stats["processing_errors"]
    + chembl_stats["too_many_atoms"]
    + chembl_stats["zero_bonds"]
)

assert (
    chembl_stats["retained"] + excluded
    == chembl_stats["total"]
), "Record accounting mismatch!"

print("\nIntermediate dataset saved to:")
print(CHEMBL_INTERMEDIATE)


CHEMBL MOLECULAR PREPROCESSING


[23:54:05] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[23:54:05] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[23:57:03] WARNING: not removing hydrogen atom without neighbors
[23:58:42] Explicit valence for atom # 13 P, 7, is greater than permitted
[23:59:04] WARNING: not removing hydrogen atom without neighbors
[00:00:04] Explicit valence for atom # 1 P, 7, is greater than permitted



CHEMBL PREPROCESSING SUMMARY
-----------------------------------------------------------------
total                    : 1,207,360
retained                 : 1,207,337
fragmented               : 52,548
zero_bonds               : 20
invalid_smiles           : 3

Retention rate: 100.00%

Sample standardized molecules:

Original : Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1
Canonical: Cc1ccc(CCN2CC(C(=O)NCc3cccnc3)CC2=O)cc1

Original : O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1
Canonical: O=C1NCCN1c1ccc(S(=O)(=O)Oc2ccc(F)cc2)cc1

Original : CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4)CC3)c12
Canonical: CCOC(=O)c1c(C)oc2nc(C)nc(NCC3CCN(Cc4ccc(F)cc4)CC3)c12

Original : Cn1cccc/c1=N\C1CC1
Canonical: Cn1cccc/c1=N\C1CC1

Original : CCCCc1ccc(NC(=S)NN2CCN(C)CC2)cc1
Canonical: CCCCc1ccc(NC(=S)NN2CCN(C)CC2)cc1

Intermediate dataset saved to:
/home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/chembl_standardized_pre_overlap.csv


## ChEMBL–BindingDB overlap removal

In [7]:

import csv
from collections import Counter

# ============================================================
# BLOCK 5: CHEMBL-BINDINGDB OVERLAP REMOVAL
# ============================================================

INPUT_FILE = CHEMBL_INTERMEDIATE

OUTPUT_FILE = (
    CHEMBL_OUTPUT / "chembl_no_bindingdb_overlap.csv"
)

assert INPUT_FILE.is_file(), (
    "Block 4 output not found. Run Block 4 first."
)

assert len(bindingdb_exclusion_smiles) > 1000, (
    "BindingDB exclusion set is unexpectedly small. "
    "Check Block 3."
)

overlap_stats = Counter()
overlap_examples = []

print("=" * 65)
print("CHEMBL-BINDINGDB OVERLAP REMOVAL")
print("=" * 65)

print(
    "BindingDB unique molecules:",
    f"{len(bindingdb_exclusion_smiles):,}"
)

# ============================================================
# STEP 1: REMOVE OVERLAPPING MOLECULES
# ============================================================

with (
    INPUT_FILE.open(
        "r", encoding="utf-8", newline=""
    ) as infile,
    OUTPUT_FILE.open(
        "w", encoding="utf-8", newline=""
    ) as outfile,
):

    reader = csv.DictReader(infile)

    writer = csv.DictWriter(
        outfile,
        fieldnames=reader.fieldnames
    )

    writer.writeheader()

    for row in reader:

        overlap_stats["total"] += 1

        smiles = row["canonical_smiles"].strip()

        # Block 4 should already have validated these
        if not smiles:
            raise ValueError(
                "Empty canonical SMILES in Block 4 output."
            )

        # Check if molecule exists in BindingDB
        if smiles in bindingdb_exclusion_smiles:

            overlap_stats["removed_overlaps"] += 1

            if len(overlap_examples) < 10:
                overlap_examples.append(smiles)

            continue

        # Keep molecules absent from BindingDB
        writer.writerow(row)
        overlap_stats["retained"] += 1


# ============================================================
# STEP 2: VERIFY ZERO REMAINING OVERLAPS
# ============================================================

remaining_overlaps = 0
verified_records = 0

with OUTPUT_FILE.open(
    "r", encoding="utf-8", newline=""
) as f:

    reader = csv.DictReader(f)

    for row in reader:

        verified_records += 1

        smiles = row["canonical_smiles"].strip()

        if smiles in bindingdb_exclusion_smiles:
            remaining_overlaps += 1


# ============================================================
# STEP 3: SUMMARY
# ============================================================

print("\nOVERLAP REMOVAL SUMMARY")
print("-" * 65)

print(
    "Total cleaned ChEMBL records:",
    f"{overlap_stats['total']:,}"
)

print(
    "Removed BindingDB overlaps:",
    f"{overlap_stats['removed_overlaps']:,}"
)

print(
    "Remaining ChEMBL records:",
    f"{overlap_stats['retained']:,}"
)

print(
    "Overlap removal percentage:",
    f"{100 * overlap_stats['removed_overlaps'] / max(overlap_stats['total'], 1):.2f}%"
)

print(
    "Remaining overlaps:",
    remaining_overlaps
)

print("\nExamples of removed molecules:")
for smiles in overlap_examples:
    print(" ", smiles)

# ============================================================
# STEP 4: INTEGRITY CHECKS
# ============================================================

assert (
    overlap_stats["total"]
    == overlap_stats["removed_overlaps"]
    + overlap_stats["retained"]
), "Record accounting mismatch!"

assert (
    verified_records == overlap_stats["retained"]
), "Output record count mismatch!"

assert remaining_overlaps == 0, (
    "ERROR: BindingDB overlaps still exist!"
)

print("\nSUCCESS: Zero canonical BindingDB overlaps remain!")

print("\nOutput saved to:")
print(OUTPUT_FILE)


CHEMBL-BINDINGDB OVERLAP REMOVAL
BindingDB unique molecules: 154,899

OVERLAP REMOVAL SUMMARY
-----------------------------------------------------------------
Total cleaned ChEMBL records: 1,207,337
Removed BindingDB overlaps: 69,688
Remaining ChEMBL records: 1,137,649
Overlap removal percentage: 5.77%
Remaining overlaps: 0

Examples of removed molecules:
  c1ccc(CN2CCC(n3cnc4cnc5[nH]ccc5c43)CC2)cc1
  CNc1ccc2ncnc(Nc3cccc(C)c3)c2c1
  CN(CC(=O)O)c1c(Br)cccc1Nc1ncnc2ccncc12
  Cc1c(C(=O)C(N)=O)c2c(OCC(=O)O)cccc2n1Cc1cccc(-c2ccccc2)c1
  C[C@@H](NC(=O)Nc1cc2[nH]nc(-c3cnn(C4COC4)c3)c2cn1)c1ccccc1
  COc1cc2c(cc1CN1CCN(C)CC1)Cc1c(-c3ccc(-c4ccc(O)cc4)cc3)n[nH]c1-2
  C=CCN(CC=C)C(=O)n1cnc(S(=O)(=O)C2CC3CCC2C3)n1
  COc1ccc(Nc2ccc3c(c2)N(c2nc4c(s2)C(=O)NC(C)(C)C4)CCO3)nn1
  N#Cc1nccc(N2C[C@@H](S(=O)(=O)c3ccccc3C(F)(F)F)C[C@H]2C(=O)NC2(C#N)CC2)n1
  COc1cc(-c2cnn(C)c2)ccc1Nc1ncc2ccnc(NCC(F)(F)F)c2n1

SUCCESS: Zero canonical BindingDB overlaps remain!

Output saved to:
/home/muhamed/repo/CTDDG/REWOR

## CHEMBL DEDUPLICATION AND FINAL CHECKS

In [8]:

import csv
from collections import Counter
from rdkit import Chem

# ============================================================
# BLOCK 6: CHEMBL DEDUPLICATION AND FINAL CHECKS
# ============================================================

INPUT_FILE = CHEMBL_OUTPUT / "chembl_no_bindingdb_overlap.csv"

CLEAN_CSV = CHEMBL_OUTPUT / "chembl_clean.csv"
CLEAN_TXT = CHEMBL_OUTPUT / "chembl_clean.txt"

assert INPUT_FILE.is_file(), "Run Block 5 first!"

seen_smiles = set()
dedup_stats = Counter()
duplicate_examples = []

print("=" * 65)
print("CHEMBL DEDUPLICATION")
print("=" * 65)

# ============================================================
# STEP 1: REMOVE DUPLICATES
# ============================================================

with (
    INPUT_FILE.open("r", encoding="utf-8", newline="") as infile,
    CLEAN_CSV.open("w", encoding="utf-8", newline="") as csvfile,
    CLEAN_TXT.open("w", encoding="utf-8") as txtfile,
):

    reader = csv.DictReader(infile)

    writer = csv.DictWriter(
        csvfile,
        fieldnames=reader.fieldnames
    )
    writer.writeheader()

    for row in reader:

        dedup_stats["total"] += 1

        smiles = row["canonical_smiles"].strip()

        # Check that the molecule is not in BindingDB
        if smiles in bindingdb_exclusion_smiles:
            raise ValueError(
                f"BindingDB overlap found: {smiles}"
            )

        # Validate saved molecular metadata
        num_atoms = int(row["num_atoms"])
        num_bonds = int(row["num_bonds"])

        if num_atoms > MAX_ATOMS or num_atoms < 1:
            raise ValueError(
                f"Invalid atom count: {smiles}"
            )

        if num_bonds < 1:
            raise ValueError(
                f"Zero-bond molecule found: {smiles}"
            )

        # Remove canonical duplicates
        if smiles in seen_smiles:

            dedup_stats["duplicates_removed"] += 1

            if len(duplicate_examples) < 5:
                duplicate_examples.append(smiles)

            continue

        seen_smiles.add(smiles)

        writer.writerow(row)
        txtfile.write(smiles + "\n")

        dedup_stats["unique_retained"] += 1


# ============================================================
# STEP 2: FINAL VERIFICATION
# ============================================================

print("\nVERIFYING CLEANED DATASET...")

assert (
    dedup_stats["total"]
    == dedup_stats["duplicates_removed"]
    + dedup_stats["unique_retained"]
), "Record count mismatch!"

assert (
    len(seen_smiles) == dedup_stats["unique_retained"]
), "Duplicate removal failed!"

assert not (
    seen_smiles & bindingdb_exclusion_smiles
), "BindingDB overlap still exists!"

# Sample molecular round-trip checks
# Full RDKit parsing was already done in Block 4.
sample_smiles = sorted(seen_smiles)[:1000]

for smiles in sample_smiles:

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(
            f"Invalid canonical SMILES: {smiles}"
        )

    reconstructed = Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True
    )

    assert reconstructed == smiles, (
        f"Canonicalization mismatch: {smiles}"
    )


# ============================================================
# STEP 3: SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FINAL CHEMBL CLEANING SUMMARY")
print("=" * 65)

print(
    "Records after BindingDB exclusion:",
    f"{dedup_stats['total']:,}"
)

print(
    "Duplicate records removed:",
    f"{dedup_stats['duplicates_removed']:,}"
)

print(
    "Final unique ChEMBL molecules:",
    f"{dedup_stats['unique_retained']:,}"
)

print(
    "Remaining BindingDB overlaps:",
    len(seen_smiles & bindingdb_exclusion_smiles)
)

print("\nDuplicate examples:")
for smiles in duplicate_examples:
    print(" ", smiles)

print("\nSaved files:")
print("CSV:", CLEAN_CSV)
print("TXT:", CLEAN_TXT)

print("\nSUCCESS: ChEMBL deduplication completed!")


CHEMBL DEDUPLICATION

VERIFYING CLEANED DATASET...

FINAL CHEMBL CLEANING SUMMARY
Records after BindingDB exclusion: 1,137,649
Duplicate records removed: 47,143
Final unique ChEMBL molecules: 1,090,506
Remaining BindingDB overlaps: 0

Duplicate examples:
  CC(C)c1nnc2n1[C@@H]1CCN(C(=O)C[C@H](N)Cc3cc(F)c(F)cc3F)C[C@@H]1CC2
  C=CCn1c(-c2ccccc2)cs\c1=N/N=C/C=C/c1ccco1
  O=[N+]([O-])c1cc2nc(O)c(O)nc2cc1CN1CCCC1
  C[S+](C)C[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1O
  CCOC(=O)COc1ccc2c(c1)CC(NCC(O)c1cccc(Cl)c1)CC2

Saved files:
CSV: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/chembl_clean.csv
TXT: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/chembl/chembl_clean.txt

SUCCESS: ChEMBL deduplication completed!


## BindingDB Protein–Ligand Pair Preprocessing

In [9]:

import csv
import gzip
from collections import Counter
from itertools import zip_longest
from rdkit import Chem

# ============================================================
# BLOCK 7: BINDINGDB PROTEIN-LIGAND PAIR PREPROCESSING
# ============================================================

print("=" * 65)
print("BINDINGDB PROTEIN-LIGAND PREPROCESSING")
print("=" * 65)

bindingdb_clean_stats = []

for split, ligand_files, protein_files in [
    ("train", train_ligand_files, train_protein_files),
    ("test", test_ligand_files, test_protein_files),
]:

    protein_map = {
        p.parent.name: p for p in protein_files
    }

    for ligand_path in ligand_files:

        fold = ligand_path.parent.name
        protein_path = protein_map[fold]

        # Separate output for every split and fold
        fold_output_dir = BINDINGDB_OUTPUT / split / fold
        fold_output_dir.mkdir(parents=True, exist_ok=True)

        output_file = fold_output_dir / "pairs_clean.csv.gz"

        stats = Counter()
        unique_proteins = set()
        unique_ligands = set()
        rejection_examples = []

        with (
            ligand_path.open(
                "r", encoding="utf-8"
            ) as ligand_file,

            protein_path.open(
                "r", encoding="utf-8"
            ) as protein_file,

            gzip.open(
                output_file,
                "wt",
                encoding="utf-8",
                newline="",
                compresslevel=4
            ) as outfile,
        ):

            writer = csv.DictWriter(
                outfile,
                fieldnames=[
                    "source_line",
                    "protein_sequence",
                    "original_smiles",
                    "canonical_smiles",
                    "num_atoms",
                    "num_bonds",
                    "fragmented",
                ],
            )

            writer.writeheader()

            for line_number, (ligand_line, protein_line) in enumerate(
                zip_longest(ligand_file, protein_file),
                start=1
            ):

                # Ensure every ligand has a protein
                if ligand_line is None or protein_line is None:
                    raise ValueError(
                        f"Line count mismatch in {fold} "
                        f"at line {line_number}"
                    )

                stats["total_pairs"] += 1

                # Reconstruct original sequences
                raw_smiles = "".join(ligand_line.split())
                protein = "".join(protein_line.split())

                # Reject incomplete pairs
                if not raw_smiles or not protein:
                    stats["empty_pairs"] += 1
                    continue

                # Validate and canonicalize ligand
                try:
                    canonical, fragmented = standardize_smiles(
                        raw_smiles
                    )

                    if canonical is None:
                        stats["invalid_smiles"] += 1
                        continue

                    mol = Chem.MolFromSmiles(canonical)

                    if mol is None:
                        stats["invalid_smiles"] += 1
                        continue

                except Exception as exc:
                    stats["processing_errors"] += 1

                    if len(rejection_examples) < 5:
                        rejection_examples.append(
                            (line_number, str(exc))
                        )
                    continue

                # Molecular constraints
                num_atoms = mol.GetNumAtoms()
                num_bonds = mol.GetNumBonds()

                if num_atoms > MAX_ATOMS:
                    stats["too_many_atoms"] += 1
                    continue

                if num_bonds == 0:
                    stats["zero_bonds"] += 1
                    continue

                if fragmented:
                    stats["fragmented_retained"] += 1

                # Save the complete aligned pair
                writer.writerow({
                    "source_line": line_number,
                    "protein_sequence": protein,
                    "original_smiles": raw_smiles,
                    "canonical_smiles": canonical,
                    "num_atoms": num_atoms,
                    "num_bonds": num_bonds,
                    "fragmented": fragmented,
                })

                stats["retained_pairs"] += 1

                unique_proteins.add(protein)
                unique_ligands.add(canonical)

        # Verify record accounting
        removed = (
            stats["empty_pairs"]
            + stats["invalid_smiles"]
            + stats["processing_errors"]
            + stats["too_many_atoms"]
            + stats["zero_bonds"]
        )

        assert (
            stats["total_pairs"]
            == stats["retained_pairs"] + removed
        ), f"Record accounting mismatch in {fold}"

        result = {
            "split": split,
            "fold": fold,
            **dict(stats),
            "unique_proteins": len(unique_proteins),
            "unique_ligands": len(unique_ligands),
            "output_file": str(output_file),
        }

        bindingdb_clean_stats.append(result)

        print(f"\n{split.upper()} | {fold}")
        print(f"  Total pairs:       {stats['total_pairs']:,}")
        print(f"  Retained pairs:    {stats['retained_pairs']:,}")
        print(f"  Invalid SMILES:    {stats['invalid_smiles']:,}")
        print(f"  Too many atoms:    {stats['too_many_atoms']:,}")
        print(f"  Zero bonds:        {stats['zero_bonds']:,}")
        print(f"  Empty pairs:       {stats['empty_pairs']:,}")
        print(f"  Processing errors: {stats['processing_errors']:,}")
        print(f"  Unique proteins:   {len(unique_proteins):,}")
        print(f"  Unique ligands:    {len(unique_ligands):,}")

        if rejection_examples:
            print("  Error examples:", rejection_examples[:3])


# ============================================================
# FINAL SUMMARY
# ============================================================

total_input = sum(
    r["total_pairs"] for r in bindingdb_clean_stats
)

total_retained = sum(
    r.get("retained_pairs", 0)
    for r in bindingdb_clean_stats
)

assert total_input == 900_502, (
    f"Unexpected BindingDB record count: {total_input}"
)

assert total_retained > 0, "No valid BindingDB pairs retained!"

print("\n" + "=" * 65)
print("BINDINGDB PREPROCESSING SUMMARY")
print("=" * 65)

print(f"Original pairs:  {total_input:,}")
print(f"Retained pairs:  {total_retained:,}")
print(f"Removed pairs:   {total_input - total_retained:,}")

print(
    f"Retention rate: "
    f"{100 * total_retained / total_input:.2f}%"
)

print("\nAll 10 folds processed successfully!")


BINDINGDB PROTEIN-LIGAND PREPROCESSING

TRAIN | train_4_org_1000_112
  Total pairs:       143,999
  Retained pairs:    143,999
  Invalid SMILES:    0
  Too many atoms:    0
  Zero bonds:        0
  Empty pairs:       0
  Processing errors: 0
  Unique proteins:   1,000
  Unique ligands:    98,622

TRAIN | train_4_org_1002_103
  Total pairs:       144,885
  Retained pairs:    144,884
  Invalid SMILES:    0
  Too many atoms:    0
  Zero bonds:        1
  Empty pairs:       0
  Processing errors: 0
  Unique proteins:   1,002
  Unique ligands:    99,375

TRAIN | train_4_org_1004_122
  Total pairs:       146,059
  Retained pairs:    146,059
  Invalid SMILES:    0
  Too many atoms:    0
  Zero bonds:        0
  Empty pairs:       0
  Processing errors: 0
  Unique proteins:   1,004
  Unique ligands:    100,237

TRAIN | train_4_org_1036_124
  Total pairs:       142,687
  Retained pairs:    142,686
  Invalid SMILES:    0
  Too many atoms:    0
  Zero bonds:        1
  Empty pairs:       0
  Proc

## BindingDB fold audit

In [10]:

import csv
import gzip
import json
from pathlib import Path

# ============================================================
# BLOCK 8: BINDINGDB FOLD AUDIT
# ============================================================

print("=" * 65)
print("BINDINGDB FOLD AUDIT")
print("=" * 65)

TRAIN_OUTPUT = BINDINGDB_OUTPUT / "train"
TEST_OUTPUT = BINDINGDB_OUTPUT / "test"

# Discover processed folds from Block 7
train_folds = {
    p.name.removeprefix("train_4_org_"): p
    for p in TRAIN_OUTPUT.glob("train_4_org_*")
    if p.is_dir()
}

test_folds = {
    p.name.removeprefix("test_4_org_"): p
    for p in TEST_OUTPUT.glob("test_4_org_*")
    if p.is_dir()
}

assert set(train_folds) == set(test_folds), (
    "Training and testing fold IDs do not match!"
)

assert len(train_folds) == 5, (
    f"Expected 5 folds, found {len(train_folds)}"
)


# ============================================================
# STEP 1: ANALYSE ONE PROCESSED FILE
# ============================================================

def analyze_fold_file(path):

    proteins = set()
    ligands = set()
    pairs = set()

    total_pairs = 0
    duplicate_pairs = 0

    with gzip.open(
        path, "rt", encoding="utf-8", newline=""
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:

            total_pairs += 1

            protein = row["protein_sequence"].strip()
            ligand = row["canonical_smiles"].strip()

            if not protein or not ligand:
                raise ValueError(
                    f"Empty protein or ligand in {path}"
                )

            proteins.add(protein)
            ligands.add(ligand)

            pair = (protein, ligand)

            if pair in pairs:
                duplicate_pairs += 1
            else:
                pairs.add(pair)

    return {
        "total_pairs": total_pairs,
        "proteins": proteins,
        "ligands": ligands,
        "pairs": pairs,
        "duplicate_pairs": duplicate_pairs,
    }


# ============================================================
# STEP 2: COMPARE TRAIN AND TEST FOR EACH FOLD
# ============================================================

fold_audit_results = []
all_unique_proteins = set()

EXPECTED_PROTEINS = {
    "1000_112": (1000, 112),
    "1002_103": (1002, 103),
    "1004_122": (1004, 122),
    "1036_124": (1036, 124),
    "1042_104": (1042, 104),
}

for fold_id in sorted(train_folds):

    train_path = (
        train_folds[fold_id] / "pairs_clean.csv.gz"
    )
    test_path = (
        test_folds[fold_id] / "pairs_clean.csv.gz"
    )

    assert train_path.is_file(), train_path
    assert test_path.is_file(), test_path

    train = analyze_fold_file(train_path)
    test = analyze_fold_file(test_path)

    # Compare train/test sets
    shared_proteins = train["proteins"] & test["proteins"]
    shared_ligands = train["ligands"] & test["ligands"]
    shared_pairs = train["pairs"] & test["pairs"]

    all_unique_proteins.update(train["proteins"])
    all_unique_proteins.update(test["proteins"])

    expected_train, expected_test = EXPECTED_PROTEINS[fold_id]

    result = {
        "fold": fold_id,
        "train_pairs": train["total_pairs"],
        "test_pairs": test["total_pairs"],

        "train_unique_proteins": len(train["proteins"]),
        "test_unique_proteins": len(test["proteins"]),

        "train_unique_ligands": len(train["ligands"]),
        "test_unique_ligands": len(test["ligands"]),

        "train_duplicate_pairs": train["duplicate_pairs"],
        "test_duplicate_pairs": test["duplicate_pairs"],

        "shared_proteins": len(shared_proteins),
        "shared_ligands": len(shared_ligands),
        "shared_pairs": len(shared_pairs),

        "expected_train_proteins": expected_train,
        "expected_test_proteins": expected_test,
    }

    fold_audit_results.append(result)

    print(f"\n{'=' * 65}")
    print(f"FOLD: {fold_id}")
    print("=" * 65)

    print("\nTRAIN:")
    print(f"  Total pairs:       {train['total_pairs']:,}")
    print(f"  Unique proteins:   {len(train['proteins']):,}")
    print(f"  Unique ligands:    {len(train['ligands']):,}")
    print(f"  Duplicate pairs:   {train['duplicate_pairs']:,}")

    print("\nTEST:")
    print(f"  Total pairs:       {test['total_pairs']:,}")
    print(f"  Unique proteins:   {len(test['proteins']):,}")
    print(f"  Unique ligands:    {len(test['ligands']):,}")
    print(f"  Duplicate pairs:   {test['duplicate_pairs']:,}")

    print("\nTRAIN-TEST OVERLAP:")
    print(f"  Shared proteins:   {len(shared_proteins):,}")
    print(f"  Shared ligands:    {len(shared_ligands):,}")
    print(f"  Shared pairs:      {len(shared_pairs):,}")

    if shared_proteins:
        print("  WARNING: Identical proteins in train and test!")

    if shared_pairs:
        print("  WARNING: Identical pairs in train and test!")

    if (
        len(train["proteins"]) != expected_train
        or len(test["proteins"]) != expected_test
    ):
        print("\n  NOTE: Protein counts differ from")
        print("  the fold's nominal train/test counts.")
        print("  We will investigate before changing splits.")

    else:
        print("\n  Protein counts match the fold name.")

    # Avoid retaining large pair sets between folds
    del train, test


# ============================================================
# STEP 3: SAVE AUDIT RESULTS
# ============================================================

AUDIT_JSON = BINDINGDB_OUTPUT / "fold_audit.json"
AUDIT_CSV = BINDINGDB_OUTPUT / "fold_audit.csv"

with AUDIT_JSON.open("w", encoding="utf-8") as f:
    json.dump(
        {
            "total_unique_proteins_across_all_folds":
                len(all_unique_proteins),
            "folds": fold_audit_results,
        },
        f,
        indent=2
    )

with AUDIT_CSV.open(
    "w", encoding="utf-8", newline=""
) as f:

    writer = csv.DictWriter(
        f, fieldnames=list(fold_audit_results[0])
    )

    writer.writeheader()
    writer.writerows(fold_audit_results)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FINAL BINDINGDB AUDIT SUMMARY")
print("=" * 65)

print(
    "Unique proteins across all 10 folds:",
    f"{len(all_unique_proteins):,}"
)

print(
    "Total shared train/test protein counts:",
    sum(r["shared_proteins"] for r in fold_audit_results)
)

print(
    "Total shared train/test pair counts:",
    sum(r["shared_pairs"] for r in fold_audit_results)
)

print("\nAudit saved to:")
print(AUDIT_JSON)
print(AUDIT_CSV)

print("\nBlock 8 completed successfully!")


BINDINGDB FOLD AUDIT

FOLD: 1000_112

TRAIN:
  Total pairs:       143,999
  Unique proteins:   1,000
  Unique ligands:    98,622
  Duplicate pairs:   27,581

TEST:
  Total pairs:       16,200
  Unique proteins:   112
  Unique ligands:    9,448
  Duplicate pairs:   2,355

TRAIN-TEST OVERLAP:
  Shared proteins:   0
  Shared ligands:    782
  Shared pairs:      0

  Protein counts match the fold name.

FOLD: 1002_103

TRAIN:
  Total pairs:       144,884
  Unique proteins:   1,002
  Unique ligands:    99,375
  Duplicate pairs:   27,624

TEST:
  Total pairs:       16,015
  Unique proteins:   103
  Unique ligands:    9,388
  Duplicate pairs:   2,480

TRAIN-TEST OVERLAP:
  Shared proteins:   0
  Shared ligands:    595
  Shared pairs:      0

  Protein counts match the fold name.

FOLD: 1004_122

TRAIN:
  Total pairs:       146,059
  Unique proteins:   1,004
  Unique ligands:    100,237
  Duplicate pairs:   27,945

TEST:
  Total pairs:       26,640
  Unique proteins:   122
  Unique ligands:   

## Protein Sequence Similarity Validation

In [11]:

import csv
import gzip
import json
import random
import hashlib

from collections import Counter, defaultdict
from Bio.Align import PairwiseAligner, substitution_matrices

# ============================================================
# BLOCK 9: PROTEIN SEQUENCE SIMILARITY VALIDATION
# ============================================================

SEED = 17
KMER_SIZE = 4
TOP_K = 3
MAX_TEST_PER_FOLD = 10  # Set None to check all test proteins
IDENTITY_THRESHOLD = 20.0

SIMILARITY_DIR = BINDINGDB_OUTPUT / "protein_similarity"
SIMILARITY_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. NEEDLEMAN-WUNSCH ALIGNER
# ============================================================

matrix = substitution_matrices.load("BLOSUM62")

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.substitution_matrix = matrix
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

VALID_AA = set(matrix.alphabet)

def normalize_sequence(seq):
    return "".join(
        aa if aa in VALID_AA else "X"
        for aa in seq.strip().upper()
    )

def sequence_id(seq):
    return hashlib.sha256(
        seq.encode("utf-8")
    ).hexdigest()[:16]

def global_identity(seq1, seq2):
    alignment = aligner.align(seq1, seq2)[0]

    a = alignment[0]
    b = alignment[1]

    matches = sum(
        x == y and x not in ("-", "X")
        for x, y in zip(a, b)
    )

    return 100.0 * matches / max(len(a), 1)

# ============================================================
# 2. LOAD UNIQUE PROTEINS
# ============================================================

def load_proteins(file_path):
    proteins = set()

    with gzip.open(
        file_path, "rt", encoding="utf-8"
    ) as f:
        reader = csv.DictReader(f)

        for row in reader:
            seq = normalize_sequence(
                row["protein_sequence"]
            )
            if seq:
                proteins.add(seq)

    return sorted(proteins)

# ============================================================
# 3. FAST CANDIDATE SCREENING
# ============================================================

def get_kmers(seq):
    return {
        seq[i:i + KMER_SIZE]
        for i in range(len(seq) - KMER_SIZE + 1)
        if "X" not in seq[i:i + KMER_SIZE]
    }

def build_index(train_proteins):
    index = defaultdict(list)
    kmer_sets = []

    for idx, protein in enumerate(train_proteins):
        kmers = get_kmers(protein)
        kmer_sets.append(kmers)

        for kmer in kmers:
            index[kmer].append(idx)

    return index, kmer_sets

def get_candidates(query, train_proteins, index, sets):
    query_kmers = get_kmers(query)
    shared = Counter()

    for kmer in query_kmers:
        shared.update(index.get(kmer, []))

    ranked = []

    for idx, intersection in shared.items():
        union = len(query_kmers) + len(sets[idx]) - intersection
        jaccard = intersection / union if union else 0
        ranked.append((jaccard, idx))

    ranked.sort(reverse=True)

    candidates = [idx for _, idx in ranked[:TOP_K]]

    # Fallback when no shared k-mers exist
    if not candidates:
        candidates = sorted(
            range(len(train_proteins)),
            key=lambda i: abs(
                len(train_proteins[i]) - len(query)
            )
        )[:TOP_K]

    return candidates

# ============================================================
# 4. ANALYSE EACH FOLD
# ============================================================

results = []
summaries = []

for fold_id in sorted(train_folds):

    train_path = train_folds[fold_id] / "pairs_clean.csv.gz"
    test_path = test_folds[fold_id] / "pairs_clean.csv.gz"

    train = load_proteins(train_path)
    test = load_proteins(test_path)

    assert not (set(train) & set(test)), (
        f"Identical train/test proteins in {fold_id}"
    )

    rng = random.Random(SEED + int(fold_id.split("_")[0]))

    if MAX_TEST_PER_FOLD is not None:
        test = sorted(
            rng.sample(test, min(MAX_TEST_PER_FOLD, len(test)))
        )

    index, train_kmers = build_index(train)

    identities = []

    print(f"\n{'=' * 55}")
    print(f"FOLD: {fold_id}")
    print(f"Training proteins: {len(train)}")
    print(f"Testing proteins analysed: {len(test)}")

    for test_seq in test:

        candidate_indices = get_candidates(
            test_seq, train, index, train_kmers
        )

        best_identity = -1.0
        best_train_seq = None

        for idx in candidate_indices:
            train_seq = train[idx]

            identity = global_identity(
                test_seq, train_seq
            )

            if identity > best_identity:
                best_identity = identity
                best_train_seq = train_seq

        if best_train_seq is None:
            raise ValueError("No training candidate found")

        identities.append(best_identity)

        results.append({
            "fold": fold_id,
            "test_protein_id": sequence_id(test_seq),
            "closest_screened_train_id": sequence_id(
                best_train_seq
            ),
            "test_length": len(test_seq),
            "train_length": len(best_train_seq),
            "sequence_identity_percent": round(
                best_identity, 4
            ),
            "above_20_percent": (
                best_identity >= IDENTITY_THRESHOLD
            ),
        })

    summary = {
        "fold": fold_id,
        "train_proteins": len(train),
        "test_proteins_checked": len(test),
        "max_screened_identity": round(
            max(identities), 2
        ),
        "mean_best_screened_identity": round(
            sum(identities) / len(identities), 2
        ),
        "test_proteins_with_screened_match_ge_20": sum(
            x >= IDENTITY_THRESHOLD for x in identities
        ),
    }

    summaries.append(summary)

    print(
        "Maximum screened identity:",
        f"{summary['max_screened_identity']:.2f}%"
    )
    print(
        "Mean best screened identity:",
        f"{summary['mean_best_screened_identity']:.2f}%"
    )
    print(
        "Test proteins with screened match >= 20%:",
        summary["test_proteins_with_screened_match_ge_20"]
    )

# ============================================================
# 5. SAVE VALIDATION RESULTS
# ============================================================

results_path = SIMILARITY_DIR / "screened_similarity.csv"
summary_path = SIMILARITY_DIR / "similarity_summary.json"

with results_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)

with summary_path.open("w", encoding="utf-8") as f:
    json.dump({
        "method": "4-mer candidate screening + Needleman-Wunsch",
        "exhaustive": False,
        "sampled_test_proteins": MAX_TEST_PER_FOLD,
        "top_candidates": TOP_K,
        "metric": "global aligned sequence identity",
        "folds": summaries,
    }, f, indent=2)

print("\n" + "=" * 55)
print("BLOCK 9 COMPLETED")
print("=" * 55)

print("Results:", results_path)
print("Summary:", summary_path)

print(
    "\nNOTE: These are screened comparisons, "
    "not exhaustive similarity guarantees."
)



FOLD: 1000_112
Training proteins: 1000
Testing proteins analysed: 10
Maximum screened identity: 16.19%
Mean best screened identity: 12.27%
Test proteins with screened match >= 20%: 0

FOLD: 1002_103
Training proteins: 1002
Testing proteins analysed: 10
Maximum screened identity: 15.23%
Mean best screened identity: 12.69%
Test proteins with screened match >= 20%: 0

FOLD: 1004_122
Training proteins: 1004
Testing proteins analysed: 10
Maximum screened identity: 15.23%
Mean best screened identity: 13.21%
Test proteins with screened match >= 20%: 0

FOLD: 1036_124
Training proteins: 1036
Testing proteins analysed: 10
Maximum screened identity: 16.19%
Mean best screened identity: 12.63%
Test proteins with screened match >= 20%: 0

FOLD: 1042_104
Training proteins: 1514
Testing proteins analysed: 10
Maximum screened identity: 99.17%
Mean best screened identity: 59.35%
Test proteins with screened match >= 20%: 7

BLOCK 9 COMPLETED
Results: /home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING

In [12]:
import pandas as pd

results = pd.read_csv(
    SIMILARITY_DIR / "screened_similarity.csv"
)

suspicious = (
    results[
        (results["fold"].astype(str) == "1042_104")
        & (results["sequence_identity_percent"] >= 20)
    ]
    .sort_values("sequence_identity_percent", ascending=False)
)

print("Suspicious protein pairs:", len(suspicious))

display(suspicious[[
    "test_protein_id",
    "closest_screened_train_id",
    "test_length",
    "train_length",
    "sequence_identity_percent"
]])

Suspicious protein pairs: 7


,test_protein_id,closest_screened_train_id,test_length,train_length,sequence_identity_percent
48,11af1fcb1bc8a20e,09f704dc979542e2,360,360,99.1667
49,aff8e650babef705,d889c8b4257efc87,1139,1139,98.2441
42,04cc920fb7ce2c8e,b40a366ad43d78aa,352,352,96.0227
40,48b7226dc34e609c,448e29b0b2d00f07,1980,2005,75.6770
45,ffb31cbc19993a54,a77a65704b9ea9bc,151,165,66.0606
46,03de8b44a3780da2,94a868d72bf8f8c5,486,414,55.8282
44,46c8efd74cce32cf,c16a6f28adbeb030,1257,1210,50.0000


## BindingDB Train/Test Split Integrity Verification

In [13]:

import csv
import gzip
import json
from itertools import zip_longest

# ============================================================
# BLOCK 10: BINDINGDB TRAIN/TEST INTEGRITY VERIFICATION
# ============================================================

print("=" * 65)
print("BINDINGDB TRAIN/TEST INTEGRITY VERIFICATION")
print("=" * 65)

# Read the saved Block 8 audit
AUDIT_PATH = BINDINGDB_OUTPUT / "fold_audit.json"

with AUDIT_PATH.open("r", encoding="utf-8") as f:
    previous_audit = json.load(f)

audit_by_fold = {
    row["fold"]: row
    for row in previous_audit["folds"]
}

# ============================================================
# STEP 1: VALIDATE PROCESSED PAIRS AGAINST RAW DATA
# ============================================================

def verify_processed_file(
    processed_path,
    raw_ligand_path,
    raw_protein_path
):
    """
    Verify every saved pair against its original
    source line in the raw BindingDB files.

    No molecular alignment or embedding computation.
    """

    stats = {
        "raw_pairs": 0,
        "retained_pairs": 0,
    }

    unique_proteins = set()
    unique_ligands = set()

    required_columns = {
        "source_line",
        "protein_sequence",
        "original_smiles",
        "canonical_smiles",
        "num_atoms",
        "num_bonds",
    }

    with (
        gzip.open(
            processed_path, "rt",
            encoding="utf-8", newline=""
        ) as cleaned_file,

        raw_ligand_path.open(
            "r", encoding="utf-8"
        ) as ligand_file,

        raw_protein_path.open(
            "r", encoding="utf-8"
        ) as protein_file,
    ):

        reader = csv.DictReader(cleaned_file)

        missing = required_columns - set(
            reader.fieldnames or []
        )

        if missing:
            raise ValueError(
                f"Missing columns in {processed_path}: {missing}"
            )

        current = next(reader, None)
        previous_source_line = 0

        for line_number, (ligand_line, protein_line) in enumerate(
            zip_longest(ligand_file, protein_file),
            start=1
        ):

            if ligand_line is None or protein_line is None:
                raise ValueError(
                    f"Raw pair mismatch at line {line_number}"
                )

            stats["raw_pairs"] += 1

            if current is None:
                continue

            source_line = int(current["source_line"])

            if source_line < line_number:
                raise ValueError(
                    f"Invalid or repeated source line: {source_line}"
                )

            if source_line != line_number:
                continue

            if source_line <= previous_source_line:
                raise ValueError(
                    f"Source lines out of order: {source_line}"
                )

            raw_smiles = "".join(ligand_line.split())
            raw_protein = "".join(protein_line.split())

            saved_smiles = current["original_smiles"]
            saved_protein = current["protein_sequence"]
            canonical = current["canonical_smiles"].strip()

            # Verify original ligand identity
            if saved_smiles != raw_smiles:
                raise ValueError(
                    f"Ligand mismatch at source line {line_number}"
                )

            # Verify original protein identity
            if saved_protein != raw_protein:
                raise ValueError(
                    f"Protein mismatch at source line {line_number}"
                )

            if not saved_protein or not canonical:
                raise ValueError(
                    f"Empty cleaned record at line {line_number}"
                )

            # Verify molecular filtering metadata
            num_atoms = int(current["num_atoms"])
            num_bonds = int(current["num_bonds"])

            if not 1 <= num_atoms <= MAX_ATOMS:
                raise ValueError(
                    f"Atom count invalid at line {line_number}"
                )

            if num_bonds < 1:
                raise ValueError(
                    f"Bond count invalid at line {line_number}"
                )

            unique_proteins.add(saved_protein)
            unique_ligands.add(canonical)

            stats["retained_pairs"] += 1
            previous_source_line = source_line

            current = next(reader, None)

        if current is not None:
            raise ValueError(
                "A saved record references a nonexistent raw line."
            )

    stats["filtered_pairs"] = (
        stats["raw_pairs"] - stats["retained_pairs"]
    )

    stats["unique_proteins"] = len(unique_proteins)
    stats["unique_ligands"] = len(unique_ligands)

    return stats, unique_proteins, unique_ligands


# ============================================================
# STEP 2: VERIFY ALL FIVE TRAIN/TEST FOLDS
# ============================================================

assert set(train_folds) == set(test_folds)
assert len(train_folds) == 5

integrity_results = []

for fold_id in sorted(train_folds):

    print(f"\n{'=' * 65}")
    print(f"FOLD: {fold_id}")
    print("=" * 65)

    split_results = {}

    for split, processed_folds, ligand_files, protein_files in [
        ("train", train_folds, train_ligand_files, train_protein_files),
        ("test", test_folds, test_ligand_files, test_protein_files),
    ]:

        fold_directory = processed_folds[fold_id]
        processed_path = fold_directory / "pairs_clean.csv.gz"

        raw_ligand_map = {
            p.parent.name: p for p in ligand_files
        }

        raw_protein_map = {
            p.parent.name: p for p in protein_files
        }

        fold_name = fold_directory.name

        raw_ligand_path = raw_ligand_map[fold_name]
        raw_protein_path = raw_protein_map[fold_name]

        stats, proteins, ligands = verify_processed_file(
            processed_path,
            raw_ligand_path,
            raw_protein_path
        )

        split_results[split] = {
            "stats": stats,
            "proteins": proteins,
            "ligands": ligands,
        }

        # Compare against Block 8
        previous = audit_by_fold[fold_id]

        assert (
            stats["retained_pairs"]
            == previous[f"{split}_pairs"]
        ), f"{fold_id}: Pair count changed since Block 8"

        assert (
            stats["unique_proteins"]
            == previous[f"{split}_unique_proteins"]
        ), f"{fold_id}: Protein count changed since Block 8"

        assert (
            stats["unique_ligands"]
            == previous[f"{split}_unique_ligands"]
        ), f"{fold_id}: Ligand count changed since Block 8"

        print(f"\n{split.upper()}")
        print(f"  Raw pairs:      {stats['raw_pairs']:,}")
        print(f"  Retained pairs: {stats['retained_pairs']:,}")
        print(f"  Filtered pairs: {stats['filtered_pairs']:,}")
        print(f"  Pair alignment: VERIFIED")

    # ========================================================
    # STEP 3: TRAIN/TEST DISJOINTNESS
    # ========================================================

    train_proteins = split_results["train"]["proteins"]
    test_proteins = split_results["test"]["proteins"]

    train_ligands = split_results["train"]["ligands"]
    test_ligands = split_results["test"]["ligands"]

    shared_proteins = train_proteins & test_proteins
    shared_ligands = train_ligands & test_ligands

    assert len(shared_proteins) == 0, (
        f"Identical train/test proteins in {fold_id}"
    )

    # Shared ligands are permitted in protein-disjoint splits.
    # No identical proteins means no identical protein-ligand
    # pairs can occur between train and test.

    fold_result = {
        "fold": fold_id,
        "train": split_results["train"]["stats"],
        "test": split_results["test"]["stats"],
        "shared_proteins": len(shared_proteins),
        "shared_ligands": len(shared_ligands),
        "raw_pair_alignment_verified": True,
        "exact_protein_disjoint": True,
    }

    integrity_results.append(fold_result)

    print("\nTRAIN/TEST CHECK")
    print(f"  Shared proteins: {len(shared_proteins)}")
    print(f"  Shared ligands:  {len(shared_ligands):,}")
    print("  Exact protein overlap: NONE")
    print("  Status: PASSED")


# ============================================================
# STEP 4: FINAL TOTALS AND SAVE REPORT
# ============================================================

total_raw = sum(
    r[split]["raw_pairs"]
    for r in integrity_results
    for split in ("train", "test")
)

total_retained = sum(
    r[split]["retained_pairs"]
    for r in integrity_results
    for split in ("train", "test")
)

assert total_raw == 900_502, (
    f"Unexpected raw BindingDB total: {total_raw}"
)

report = {
    "total_raw_pairs": total_raw,
    "total_retained_pairs": total_retained,
    "total_filtered_pairs": total_raw - total_retained,
    "exact_protein_overlap_check": "passed",
    "raw_pair_alignment_check": "passed",
    "protein_similarity_check": "not performed in Block 10",
    "folds": integrity_results,
}

REPORT_FILE = (
    BINDINGDB_OUTPUT / "split_integrity_report.json"
)

with REPORT_FILE.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n" + "=" * 65)
print("BLOCK 10 — FINAL SUMMARY")
print("=" * 65)

print(f"Raw BindingDB pairs:      {total_raw:,}")
print(f"Retained pairs:           {total_retained:,}")
print(f"Filtered pairs:           {total_raw - total_retained:,}")
print("Protein-ligand alignment: PASSED")
print("Exact train/test overlap: PASSED")
print("All five folds:           VERIFIED")

print("\nReport saved to:")
print(REPORT_FILE)

print("\nBLOCK 10 COMPLETED SUCCESSFULLY!")


BINDINGDB TRAIN/TEST INTEGRITY VERIFICATION

FOLD: 1000_112

TRAIN
  Raw pairs:      143,999
  Retained pairs: 143,999
  Filtered pairs: 0
  Pair alignment: VERIFIED

TEST
  Raw pairs:      16,200
  Retained pairs: 16,200
  Filtered pairs: 0
  Pair alignment: VERIFIED

TRAIN/TEST CHECK
  Shared proteins: 0
  Shared ligands:  782
  Exact protein overlap: NONE
  Status: PASSED

FOLD: 1002_103

TRAIN
  Raw pairs:      144,885
  Retained pairs: 144,884
  Filtered pairs: 1
  Pair alignment: VERIFIED

TEST
  Raw pairs:      16,015
  Retained pairs: 16,015
  Filtered pairs: 0
  Pair alignment: VERIFIED

TRAIN/TEST CHECK
  Shared proteins: 0
  Shared ligands:  595
  Exact protein overlap: NONE
  Status: PASSED

FOLD: 1004_122

TRAIN
  Raw pairs:      146,059
  Retained pairs: 146,059
  Filtered pairs: 0
  Pair alignment: VERIFIED

TEST
  Raw pairs:      26,640
  Retained pairs: 26,640
  Filtered pairs: 0
  Pair alignment: VERIFIED

TRAIN/TEST CHECK
  Shared proteins: 0
  Shared ligands:  1,044

## CTDDG Atom-Type Vocabulary Construction

In [14]:

import csv
import gzip
import json
from collections import Counter
from pathlib import Path
from rdkit import Chem

# ============================================================
# BLOCK 11: CTDDG ATOM-TYPE VOCABULARY CONSTRUCTION
# ============================================================

print("=" * 65)
print("CTDDG ATOM-TYPE VOCABULARY")
print("=" * 65)

VOCAB_DIR = OUTPUT_DIR / "vocabulary"
VOCAB_DIR.mkdir(parents=True, exist_ok=True)

CHEMBL_CLEAN_FILE = CHEMBL_OUTPUT / "chembl_clean.csv"

TRAIN_FILES = sorted(
    (BINDINGDB_OUTPUT / "train").rglob("pairs_clean.csv.gz")
)

TEST_FILES = sorted(
    (BINDINGDB_OUTPUT / "test").rglob("pairs_clean.csv.gz")
)

assert CHEMBL_CLEAN_FILE.is_file()
assert len(TRAIN_FILES) == 5
assert len(TEST_FILES) == 5

# ============================================================
# STEP 1: DEFINE ATOM TYPES
# ============================================================

def get_atom_type(atom):
    """
    CTDDG reference-style atom representation:

    (element symbol, formal charge, explicit H count)
    """
    return (
        atom.GetSymbol(),
        int(atom.GetFormalCharge()),
        int(atom.GetNumExplicitHs())
    )


def get_molecule_atom_types(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    return [
        get_atom_type(atom)
        for atom in mol.GetAtoms()
    ]


# ============================================================
# STEP 2: COLLECT ATOM TYPES FROM CLEANED CHEMBL
# ============================================================

atom_counts = Counter()
molecule_type_counts = Counter()

chembl_molecules = 0

print("\nScanning cleaned ChEMBL...")

with CHEMBL_CLEAN_FILE.open(
    "r", encoding="utf-8", newline=""
) as f:

    reader = csv.DictReader(f)

    assert "canonical_smiles" in reader.fieldnames

    for row in reader:

        smiles = row["canonical_smiles"].strip()
        types = get_molecule_atom_types(smiles)

        atom_counts.update(types)
        molecule_type_counts.update(set(types))

        chembl_molecules += 1

        if chembl_molecules % 250_000 == 0:
            print(
                f"  Processed {chembl_molecules:,} molecules"
            )

chembl_atom_types = set(atom_counts)

print("\nChEMBL molecules:", f"{chembl_molecules:,}")
print("ChEMBL atom types:", len(chembl_atom_types))


# ============================================================
# STEP 3: INCLUDE BINDINGDB TRAINING ATOM TYPES
# ============================================================

print("\nScanning BindingDB training ligands...")

seen_train_smiles = set()

train_atom_types = set()

for file_path in TRAIN_FILES:

    file_unique = 0

    with gzip.open(
        file_path, "rt",
        encoding="utf-8", newline=""
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:

            smiles = row["canonical_smiles"].strip()

            # A ligand can occur with multiple proteins
            # and across multiple folds.
            # Count each unique molecular structure once.
            if smiles in seen_train_smiles:
                continue

            seen_train_smiles.add(smiles)
            file_unique += 1

            types = get_molecule_atom_types(smiles)

            atom_counts.update(types)
            molecule_type_counts.update(set(types))

            train_atom_types.update(types)

    print(
        f"  {file_path.parent.name}: "
        f"{file_unique:,} new unique ligands"
    )

print(
    "\nUnique BindingDB training ligands:",
    f"{len(seen_train_smiles):,}"
)

print(
    "BindingDB training atom types:",
    len(train_atom_types)
)

print(
    "New atom types beyond ChEMBL:",
    len(train_atom_types - chembl_atom_types)
)


# ============================================================
# STEP 4: CREATE DETERMINISTIC VOCABULARY
# ============================================================

# Sorting ensures the same atom types always receive
# the same IDs, independent of file reading order.

atom_types = sorted(atom_counts.keys())

ATOM_TO_INDEX = {
    atom_type: idx
    for idx, atom_type in enumerate(atom_types)
}

INDEX_TO_ATOM = {
    idx: atom_type
    for atom_type, idx in ATOM_TO_INDEX.items()
}

NUM_ATOM_TYPES = len(atom_types)

assert len(ATOM_TO_INDEX) == len(INDEX_TO_ATOM)
assert sorted(INDEX_TO_ATOM) == list(range(NUM_ATOM_TYPES))

print("\n" + "=" * 65)
print("FINAL ATOM VOCABULARY")
print("=" * 65)

print("Total atom types:", NUM_ATOM_TYPES)

for atom_type, idx in ATOM_TO_INDEX.items():
    print(
        f"  ID {idx:>3}: {atom_type}"
        f" | atoms: {atom_counts[atom_type]:,}"
    )


# ============================================================
# STEP 5: CHECK BINDINGDB TEST COVERAGE
# ============================================================

print("\nChecking BindingDB test atom types...")

seen_test_smiles = set()

test_atom_types = set()
test_oov_molecules = 0
oov_types = Counter()

for file_path in TEST_FILES:

    with gzip.open(
        file_path, "rt",
        encoding="utf-8", newline=""
    ) as f:

        reader = csv.DictReader(f)

        for row in reader:

            smiles = row["canonical_smiles"].strip()

            if smiles in seen_test_smiles:
                continue

            seen_test_smiles.add(smiles)

            types = set(
                get_molecule_atom_types(smiles)
            )

            test_atom_types.update(types)

            missing = types - set(ATOM_TO_INDEX)

            if missing:
                test_oov_molecules += 1
                oov_types.update(missing)

print(
    "Unique test ligands:",
    f"{len(seen_test_smiles):,}"
)

print(
    "Test atom types:",
    len(test_atom_types)
)

print(
    "Unseen test atom types:",
    len(oov_types)
)

print(
    "Test molecules with unseen atom types:",
    f"{test_oov_molecules:,}"
)

if oov_types:
    print("\nWARNING: Unseen test atom types detected!")
    for atom_type, count in sorted(oov_types.items()):
        print(f"  {atom_type}: {count} molecules")
else:
    print("\nAll test atom types are covered!")


# ============================================================
# STEP 6: SAVE VOCABULARY
# ============================================================

VOCAB_FILE = VOCAB_DIR / "atom_vocabulary.json"

vocabulary = {
    "definition": [
        "element_symbol",
        "formal_charge",
        "explicit_hydrogens"
    ],
    "index_base": 0,
    "num_atom_types": NUM_ATOM_TYPES,
    "sources": [
        "chembl_clean.csv",
        "bindingdb_train"
    ],
    "atom_types": [
        {
            "id": idx,
            "symbol": atom_type[0],
            "formal_charge": atom_type[1],
            "explicit_hydrogens": atom_type[2],
            "atom_occurrences": atom_counts[atom_type],
            "molecule_occurrences": (
                molecule_type_counts[atom_type]
            )
        }
        for idx, atom_type in INDEX_TO_ATOM.items()
    ]
}

with VOCAB_FILE.open(
    "w", encoding="utf-8"
) as f:
    json.dump(vocabulary, f, indent=2)


# ============================================================
# STEP 7: SAVE TEST COVERAGE REPORT
# ============================================================

COVERAGE_FILE = VOCAB_DIR / "atom_coverage_report.json"

coverage_report = {
    "chembl_molecules": chembl_molecules,
    "unique_bindingdb_train_ligands": len(
        seen_train_smiles
    ),
    "unique_bindingdb_test_ligands": len(
        seen_test_smiles
    ),
    "chembl_atom_types": len(chembl_atom_types),
    "bindingdb_train_atom_types": len(
        train_atom_types
    ),
    "final_vocabulary_size": NUM_ATOM_TYPES,
    "test_oov_molecules": test_oov_molecules,
    "test_oov_atom_types": [
        {
            "symbol": atom_type[0],
            "formal_charge": atom_type[1],
            "explicit_hydrogens": atom_type[2],
            "molecules": count
        }
        for atom_type, count in sorted(oov_types.items())
    ]
}

with COVERAGE_FILE.open(
    "w", encoding="utf-8"
) as f:
    json.dump(coverage_report, f, indent=2)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("BLOCK 11 — SUMMARY")
print("=" * 65)

print(f"ChEMBL molecules: {chembl_molecules:,}")
print(
    "Unique BindingDB training ligands:",
    f"{len(seen_train_smiles):,}"
)
print(f"Final atom vocabulary size: {NUM_ATOM_TYPES}")
print(f"Unseen test atom types: {len(oov_types)}")

print("\nSaved:")
print(VOCAB_FILE)
print(COVERAGE_FILE)

print("\nBLOCK 11 COMPLETED!")


CTDDG ATOM-TYPE VOCABULARY

Scanning cleaned ChEMBL...
  Processed 250,000 molecules
  Processed 500,000 molecules
  Processed 750,000 molecules
  Processed 1,000,000 molecules

ChEMBL molecules: 1,090,506
ChEMBL atom types: 35

Scanning BindingDB training ligands...
  train_4_org_1000_112: 98,622 new unique ligands
  train_4_org_1002_103: 4,349 new unique ligands
  train_4_org_1004_122: 710 new unique ligands
  train_4_org_1036_124: 458 new unique ligands
  train_4_org_1042_104: 49,540 new unique ligands

Unique BindingDB training ligands: 153,679
BindingDB training atom types: 50
New atom types beyond ChEMBL: 20

FINAL ATOM VOCABULARY
Total atom types: 55
  ID   0: ('B', -1, 0) | atoms: 16
  ID   1: ('B', -1, 3) | atoms: 4
  ID   2: ('B', 0, 0) | atoms: 225
  ID   3: ('Br', 0, 0) | atoms: 50,426
  ID   4: ('C', -1, 0) | atoms: 761
  ID   5: ('C', -1, 1) | atoms: 23
  ID   6: ('C', 0, 0) | atoms: 22,641,974
  ID   7: ('C', 0, 1) | atoms: 473,641
  ID   8: ('C', 0, 2) | atoms: 21
  ID 

## CTDDG Molecular Graph Encoding and Vocabulary Compatibility Check

In [15]:

import csv
import gzip
import json
from itertools import islice
from pathlib import Path
from collections import Counter
from rdkit import Chem

# ============================================================
# BLOCK 12: GRAPH ENCODING AND VOCABULARY COMPATIBILITY
# ============================================================

ROOT = Path("/home/muhamed/repo/CTDDG/REWORKED_CODE")
OUT = ROOT / "PREPROCESSING_OUTPUT"

VOCAB_FILE = OUT / "vocabulary" / "atom_vocabulary.json"
CHEMBL_FILE = OUT / "chembl" / "chembl_clean.csv"
BINDINGDB_DIR = OUT / "bindingdb"

AUDIT_DIR = OUT / "graph_compatibility"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Keep this lightweight
CHEMBL_SAMPLE = 300
BINDINGDB_SAMPLE_PER_FILE = 60

# ============================================================
# 1. LOAD OUR ATOM VOCABULARY
# ============================================================

with VOCAB_FILE.open("r", encoding="utf-8") as f:
    vocab_data = json.load(f)

entries = vocab_data["atom_types"]

assert [x["id"] for x in entries] == list(range(len(entries)))

atom_types = [
    (
        x["symbol"],
        int(x["formal_charge"]),
        int(x["explicit_hydrogens"])
    )
    for x in entries
]

assert len(atom_types) == len(set(atom_types))

atom_to_id = {
    atom_type: i
    for i, atom_type in enumerate(atom_types)
}

id_to_atom = {
    i: atom_type
    for atom_type, i in atom_to_id.items()
}

# Match the four bond types used by CTDDG
bond_types = [
    Chem.BondType.AROMATIC,
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE
]

bond_to_id = {
    bond_type: i
    for i, bond_type in enumerate(bond_types)
}

print("=" * 65)
print("CTDDG GRAPH COMPATIBILITY AUDIT")
print("=" * 65)

print("Atom types:", len(atom_types))
print("Bond types:", len(bond_types))

# ============================================================
# 2. GRAPH ENCODER
# ============================================================

def encode_molecule(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    node_ids = []
    edge_indices = []
    edge_types = []

    # Encode every atom
    for atom in mol.GetAtoms():

        key = (
            atom.GetSymbol(),
            int(atom.GetFormalCharge()),
            int(atom.GetNumExplicitHs())
        )

        if key not in atom_to_id:
            raise ValueError(f"Unknown atom type: {key}")

        atom_id = atom_to_id[key]

        assert id_to_atom[atom_id] == key

        node_ids.append(atom_id)

    # Encode every bond
    for bond in mol.GetBonds():

        bond_type = bond.GetBondType()

        if bond_type not in bond_to_id:
            raise ValueError(
                f"Unsupported bond type: {bond_type}"
            )

        edge_indices.append([
            bond.GetBeginAtomIdx(),
            bond.GetEndAtomIdx()
        ])

        edge_types.append(bond_to_id[bond_type])

    # Canonical ranks used by reference graph code
    ranks = list(Chem.CanonicalRankAtoms(mol))

    assert len(node_ids) == mol.GetNumAtoms()
    assert len(edge_indices) == mol.GetNumBonds()
    assert len(edge_types) == len(edge_indices)
    assert len(ranks) == len(node_ids)

    # Check stereochemistry present in source molecule
    has_stereo = (
        any(
            a.GetChiralTag() != Chem.ChiralType.CHI_UNSPECIFIED
            for a in mol.GetAtoms()
        )
        or any(
            b.GetStereo() != Chem.BondStereo.STEREONONE
            for b in mol.GetBonds()
        )
    )

    return {
        "node_ids": node_ids,
        "edge_indices": edge_indices,
        "edge_types": edge_types,
        "atom_ranks": ranks,
        "num_atoms": mol.GetNumAtoms(),
        "num_bonds": mol.GetNumBonds(),
        "has_stereochemistry": has_stereo,
    }


# ============================================================
# 3. TEST GRAPH ENCODING ON CLEANED DATA
# ============================================================

stats = Counter()
failures = []
used_atom_ids = set()
used_bond_ids = set()

def test_smiles(smiles, source):
    stats["tested"] += 1

    try:
        graph = encode_molecule(smiles)

        assert 1 <= graph["num_atoms"] <= 100
        assert graph["num_bonds"] >= 1

        assert all(
            0 <= i < len(atom_types)
            for i in graph["node_ids"]
        )

        assert all(
            0 <= i < len(bond_types)
            for i in graph["edge_types"]
        )

        used_atom_ids.update(graph["node_ids"])
        used_bond_ids.update(graph["edge_types"])

        stats["passed"] += 1

        if graph["has_stereochemistry"]:
            stats["stereo_molecules"] += 1

    except Exception as exc:
        stats["failed"] += 1

        if len(failures) < 20:
            failures.append({
                "source": source,
                "smiles": smiles,
                "error": str(exc)
            })


print("\nTesting ChEMBL molecules...")

with CHEMBL_FILE.open(
    "r", encoding="utf-8", newline=""
) as f:

    reader = csv.DictReader(f)

    for row in islice(reader, CHEMBL_SAMPLE):
        test_smiles(row["canonical_smiles"], "chembl")

print("\nTesting BindingDB molecules...")

processed_files = sorted(
    BINDINGDB_DIR.rglob("pairs_clean.csv.gz")
)

assert len(processed_files) == 10, (
    f"Expected 10 BindingDB files, found {len(processed_files)}"
)

for file_path in processed_files:

    before = stats["tested"]

    with gzip.open(
        file_path, "rt",
        encoding="utf-8", newline=""
    ) as f:

        reader = csv.DictReader(f)

        for row in islice(
            reader,
            BINDINGDB_SAMPLE_PER_FILE
        ):
            test_smiles(
                row["canonical_smiles"],
                file_path.parent.name
            )

    print(
        f"  {file_path.parent.name}: "
        f"{stats['tested'] - before} checked"
    )


# ============================================================
# 4. COMPARE AGAINST EXISTING MODEL VOCABULARIES
# ============================================================

def load_legacy_atom_types(path):
    result = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:

            if not line.strip():
                continue

            parts = line.strip().split(",")

            if len(parts) != 3:
                raise ValueError(
                    f"Invalid atom type in {path}: {line}"
                )

            result.append((
                parts[0],
                int(parts[1]),
                int(parts[2])
            ))

    if len(result) != len(set(result)):
        raise ValueError(
            f"Duplicate atom types in {path}"
        )

    return result


model_vocab_candidates = [
    ROOT / "model_weights" / "nitesh_model" / "atom_types.txt",
    ROOT / "chembl_preprocessed" / "atom_types.txt",
]

vocab_comparisons = []

print("\n" + "=" * 65)
print("EXISTING MODEL VOCABULARY COMPARISON")
print("=" * 65)

for path in model_vocab_candidates:

    if not path.is_file():
        print(f"\nNot found: {path}")
        continue

    legacy = load_legacy_atom_types(path)

    legacy_lookup = {
        atom: i
        for i, atom in enumerate(legacy)
    }

    missing = sorted(set(atom_types) - set(legacy))
    extra = sorted(set(legacy) - set(atom_types))

    different_ids = [
        atom
        for atom in atom_types
        if atom in legacy_lookup
        and atom_to_id[atom] != legacy_lookup[atom]
    ]

    result = {
        "path": str(path),
        "model_atom_types": len(legacy),
        "new_atom_types": len(atom_types),
        "missing_from_model": missing,
        "extra_in_model": extra,
        "shared_types_with_different_ids": len(different_ids),
        "exact_order_match": legacy == atom_types,
    }

    vocab_comparisons.append(result)

    print(f"\nFile: {path}")
    print("Model vocabulary size:", len(legacy))
    print("New vocabulary size:", len(atom_types))
    print("Types absent from model:", len(missing))
    print("Extra model types:", len(extra))
    print("Shared types with different IDs:", len(different_ids))
    print("Exact ordering match:", legacy == atom_types)


# ============================================================
# 5. CHECK AVAILABLE MODEL CONFIGURATION METADATA
# ============================================================

model_meta_path = (
    ROOT / "pretraining"
    / "pretraining_runs"
    / "ctddg_torch_eq8_bs32_v1"
    / "run_meta.json"
)

model_config_summary = None

if model_meta_path.is_file():

    with model_meta_path.open("r", encoding="utf-8") as f:
        meta = json.load(f)

    config = meta.get("model_config", {})

    model_config_summary = {
        "path": str(model_meta_path),
        "n_atom": config.get("n_atom"),
        "n_bond": config.get("n_bond"),
    }

    print("\nExisting model configuration:")
    print(model_config_summary)

    if config.get("n_atom") is not None:
        if int(config["n_atom"]) != len(atom_types):
            print(
                "WARNING: Model atom count differs "
                "from our new vocabulary."
            )


# ============================================================
# 6. SAVE REPORT
# ============================================================

report = {
    "new_atom_vocab_size": len(atom_types),
    "bond_vocab_size": len(bond_types),
    "sampled_molecules": dict(stats),
    "used_atom_types_in_sample": len(used_atom_ids),
    "used_bond_types_in_sample": len(used_bond_ids),
    "failures": failures,
    "model_vocab_comparisons": vocab_comparisons,
    "model_config": model_config_summary,
    "sample_only": True,
    "stereochemistry_encoded": False,
}

report_path = AUDIT_DIR / "graph_compatibility_report.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("BLOCK 12 — SUMMARY")
print("=" * 65)

print(f"Molecules tested: {stats['tested']:,}")
print(f"Successfully encoded: {stats['passed']:,}")
print(f"Failed: {stats['failed']:,}")
print(f"Atom types used in sample: {len(used_atom_ids)}")
print(f"Bond types used in sample: {len(used_bond_ids)}")
print(
    "Sample molecules containing stereochemistry:",
    stats["stereo_molecules"]
)

if failures:
    print("\nFirst failure examples:")
    for item in failures[:5]:
        print(item)
else:
    print("\nAll sampled molecular graphs encoded successfully!")

print("\nReport saved to:")
print(report_path)

print("\nBLOCK 12 COMPLETED!")


CTDDG GRAPH COMPATIBILITY AUDIT
Atom types: 55
Bond types: 4

Testing ChEMBL molecules...

Testing BindingDB molecules...
  test_4_org_1000_112: 60 checked
  test_4_org_1002_103: 60 checked
  test_4_org_1004_122: 60 checked
  test_4_org_1036_124: 60 checked
  test_4_org_1042_104: 60 checked
  train_4_org_1000_112: 60 checked
  train_4_org_1002_103: 60 checked
  train_4_org_1004_122: 60 checked
  train_4_org_1036_124: 60 checked
  train_4_org_1042_104: 60 checked

EXISTING MODEL VOCABULARY COMPARISON

File: /home/muhamed/repo/CTDDG/REWORKED_CODE/model_weights/nitesh_model/atom_types.txt
Model vocabulary size: 67
New vocabulary size: 55
Types absent from model: 0
Extra model types: 12
Shared types with different IDs: 52
Exact ordering match: False

File: /home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/atom_types.txt
Model vocabulary size: 43
New vocabulary size: 55
Types absent from model: 18
Extra model types: 6
Shared types with different IDs: 37
Exact ordering match: False


## Final Dataset Preparation and Export

In [16]:

import csv
import json
import hashlib
from pathlib import Path
from itertools import zip_longest

# ============================================================
# BLOCK 13: FINAL DATASET PREPARATION AND EXPORT
# ============================================================

ROOT = Path("/home/muhamed/repo/CTDDG/REWORKED_CODE")
OUT = ROOT / "PREPROCESSING_OUTPUT"

CHEMBL_DIR = OUT / "chembl"
BINDINGDB_DIR = OUT / "bindingdb"
VOCAB_DIR = OUT / "vocabulary"

CHEMBL_CSV = CHEMBL_DIR / "chembl_clean.csv"
CHEMBL_TXT = CHEMBL_DIR / "chembl_clean.txt"

VOCAB_JSON = VOCAB_DIR / "atom_vocabulary.json"
ATOM_TYPES_TXT = VOCAB_DIR / "atom_types.txt"

GRAPH_REPORT = (
    OUT / "graph_compatibility" /
    "graph_compatibility_report.json"
)

INTEGRITY_REPORT = (
    BINDINGDB_DIR / "split_integrity_report.json"
)

MANIFEST_FILE = OUT / "preprocessing_manifest.json"

print("=" * 65)
print("BLOCK 13: FINAL DATASET PREPARATION")
print("=" * 65)

# ============================================================
# STEP 1: CHECK REQUIRED FILES
# ============================================================

required_files = [
    CHEMBL_CSV,
    CHEMBL_TXT,
    VOCAB_JSON,
    GRAPH_REPORT,
    INTEGRITY_REPORT,
]

for path in required_files:
    assert path.is_file(), f"Missing file: {path}"

print("\nAll required files found!")

# ============================================================
# STEP 2: VERIFY CHEMBL CSV AND TXT MATCH EXACTLY
# ============================================================

print("\nVerifying ChEMBL CSV/TXT consistency...")

chembl_count = 0

with (
    CHEMBL_CSV.open("r", encoding="utf-8", newline="") as cf,
    CHEMBL_TXT.open("r", encoding="utf-8") as tf,
):
    reader = csv.DictReader(cf)

    assert "canonical_smiles" in reader.fieldnames

    for row, txt_line in zip_longest(reader, tf):

        if row is None or txt_line is None:
            raise ValueError(
                "ChEMBL CSV/TXT have different record counts!"
            )

        csv_smiles = row["canonical_smiles"].strip()
        txt_smiles = txt_line.strip()

        if csv_smiles != txt_smiles:
            raise ValueError(
                f"SMILES mismatch at record {chembl_count + 1}"
            )

        if not csv_smiles:
            raise ValueError("Empty ChEMBL SMILES detected")

        chembl_count += 1

print(f"ChEMBL molecules verified: {chembl_count:,}")

# ============================================================
# STEP 3: LOAD AND EXPORT THE ATOM VOCABULARY
# ============================================================

print("\nExporting atom_types.txt...")

with VOCAB_JSON.open("r", encoding="utf-8") as f:
    vocab = json.load(f)

entries = vocab["atom_types"]

assert [entry["id"] for entry in entries] == list(
    range(len(entries))
)

atom_types = [
    (
        entry["symbol"],
        int(entry["formal_charge"]),
        int(entry["explicit_hydrogens"])
    )
    for entry in entries
]

assert len(atom_types) == len(set(atom_types))
assert len(atom_types) == vocab["num_atom_types"]

# Write the format expected by the CTDDG MoleculeSpec loader
with ATOM_TYPES_TXT.open("w", encoding="utf-8") as f:
    for symbol, charge, hydrogens in atom_types:
        f.write(f"{symbol},{charge},{hydrogens}\n")

# Verify the exported ordering
with ATOM_TYPES_TXT.open("r", encoding="utf-8") as f:
    exported = [
        (
            parts[0],
            int(parts[1]),
            int(parts[2])
        )
        for line in f if line.strip()
        for parts in [line.strip().split(",")]
    ]

assert exported == atom_types

vocab_sha256 = hashlib.sha256(
    ATOM_TYPES_TXT.read_bytes()
).hexdigest()

print(f"Atom types exported: {len(atom_types)}")
print(f"Vocabulary SHA256: {vocab_sha256}")

# ============================================================
# STEP 4: CHECK PREVIOUS AUDIT RESULTS
# ============================================================

with GRAPH_REPORT.open("r", encoding="utf-8") as f:
    graph_report = json.load(f)

with INTEGRITY_REPORT.open("r", encoding="utf-8") as f:
    integrity = json.load(f)

graph_failures = graph_report["sampled_molecules"].get(
    "failed", 0
)

assert graph_failures == 0, (
    f"Graph encoding had {graph_failures} failures. "
    "Investigate Block 12 before proceeding."
)

assert integrity["raw_pair_alignment_check"] == "passed"
assert integrity["exact_protein_overlap_check"] == "passed"

bindingdb_pairs = integrity["total_retained_pairs"]

print("\nPrevious integrity checks: PASSED")
print(f"BindingDB retained pairs: {bindingdb_pairs:,}")

# ============================================================
# STEP 5: EXISTING CHECKPOINT COMPATIBILITY
# ============================================================

compatibility = []

for item in graph_report.get("model_vocab_comparisons", []):
    exact_match = item["exact_order_match"]

    compatibility.append({
        "model_vocab_path": item["path"],
        "exact_vocabulary_match": exact_match,
        "model_vocab_size": item["model_atom_types"],
    })

    if not exact_match:
        print(
            "\nWARNING: Existing model vocabulary differs:"
        )
        print(item["path"])
        print(
            "Do NOT use this new vocabulary directly "
            "with that model's pretrained weights."
        )

model_config = graph_report.get("model_config")

if model_config and model_config.get("n_atom") is not None:
    if int(model_config["n_atom"]) != len(atom_types):
        print(
            "\nWARNING: Existing model's n_atom does "
            "not match our new vocabulary size."
        )

# ============================================================
# STEP 6: CREATE DATASET MANIFEST
# ============================================================

manifest = {
    "preprocessing_version": 1,
    "chembl": {
        "csv": str(CHEMBL_CSV),
        "txt": str(CHEMBL_TXT),
        "molecule_count": chembl_count,
        "csv_txt_consistent": True,
        "ames_labels_added": False,
    },
    "bindingdb": {
        "output_directory": str(BINDINGDB_DIR),
        "original_pairs": integrity["total_raw_pairs"],
        "retained_pairs": bindingdb_pairs,
        "filtered_pairs": integrity["total_filtered_pairs"],
        "protein_ligand_alignment_verified": True,
        "exact_train_test_protein_overlap": 0,
    },
    "vocabulary": {
        "json": str(VOCAB_JSON),
        "txt": str(ATOM_TYPES_TXT),
        "num_atom_types": len(atom_types),
        "num_bond_types": 4,
        "sha256": vocab_sha256,
    },
    "graph_audit": {
        "sampled_molecules": graph_report[
            "sampled_molecules"
        ],
        "full_graph_encoding_verified": False,
        "model_compatibility": compatibility,
    },
}

with MANIFEST_FILE.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("BLOCK 13 — FINAL SUMMARY")
print("=" * 65)

print(f"Cleaned ChEMBL molecules:  {chembl_count:,}")
print(f"Cleaned BindingDB pairs:   {bindingdb_pairs:,}")
print(f"Atom vocabulary size:     {len(atom_types)}")
print("ChEMBL CSV/TXT consistency: PASSED")
print("Graph sample encoding:     PASSED")
print("BindingDB alignment:       PASSED")

print("\nFiles created:")
print(ATOM_TYPES_TXT)
print(MANIFEST_FILE)

print("\nBLOCK 13 COMPLETED!")


BLOCK 13: FINAL DATASET PREPARATION

All required files found!

Verifying ChEMBL CSV/TXT consistency...
ChEMBL molecules verified: 1,090,506

Exporting atom_types.txt...
Atom types exported: 55
Vocabulary SHA256: 3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f

Previous integrity checks: PASSED
BindingDB retained pairs: 900,499

/home/muhamed/repo/CTDDG/REWORKED_CODE/model_weights/nitesh_model/atom_types.txt
Do NOT use this new vocabulary directly with that model's pretrained weights.

/home/muhamed/repo/CTDDG/REWORKED_CODE/chembl_preprocessed/atom_types.txt
Do NOT use this new vocabulary directly with that model's pretrained weights.


BLOCK 13 — FINAL SUMMARY
Cleaned ChEMBL molecules:  1,090,506
Cleaned BindingDB pairs:   900,499
Atom vocabulary size:     55
ChEMBL CSV/TXT consistency: PASSED
Graph sample encoding:     PASSED
BindingDB alignment:       PASSED

Files created:
/home/muhamed/repo/CTDDG/REWORKED_CODE/PREPROCESSING_OUTPUT/vocabulary/atom_types.txt
/home/m

## Random Forest Ames Mutagenicity Labelling

In [2]:

import csv
import json
import os
import hashlib
from pathlib import Path
from collections import Counter
from itertools import islice

import joblib
import numpy as np
import sklearn
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import rdFingerprintGenerator

# ============================================================
# BLOCK 14: RANDOM FOREST AMES MUTAGENICITY LABELLING
# ============================================================

ROOT = Path("/home/muhamed/repo/CTDDG/REWORKED_CODE")
CHEMBL_DIR = ROOT / "PREPROCESSING_OUTPUT" / "chembl"

INPUT_CSV = CHEMBL_DIR / "chembl_clean.csv"

RF_PATH = ROOT / "Random_Forest" / "RF_NItesh.joblib"

OUTPUT_CSV = CHEMBL_DIR / "chembl_ames_predicted.csv"
OUTPUT_TXT = CHEMBL_DIR / "chembl_final.txt"
REPORT_PATH = CHEMBL_DIR / "ames_labelling_report.json"

BATCH_SIZE = 512
FP_RADIUS = 2
FP_BITS = 2048
EXPECTED_COUNT = 1_090_506

assert INPUT_CSV.is_file(), f"Missing ChEMBL: {INPUT_CSV}"
assert RF_PATH.is_file(), f"Missing Random Forest: {RF_PATH}"

print("=" * 65)
print("BLOCK 14: RANDOM FOREST AMES LABELLING")
print("=" * 65)

# ============================================================
# STEP 1: LOAD TRUSTED RANDOM FOREST
# ============================================================

# joblib uses pickle. Load only a trusted local model.
rf = joblib.load(RF_PATH)

classes = list(np.asarray(rf.classes_).tolist())

assert set(classes) == {0, 1}, (
    f"Unexpected RF classes: {classes}"
)

if hasattr(rf, "n_features_in_"):
    assert int(rf.n_features_in_) == FP_BITS, (
        "RF fingerprint dimension mismatch!"
    )

assert hasattr(rf, "predict")
assert hasattr(rf, "predict_proba")

positive_idx = classes.index(1)

print("\nRandom Forest loaded successfully")
print("Model:", type(rf).__name__)
print("Classes:", classes)
print("Input features:", getattr(rf, "n_features_in_", "unknown"))
print("scikit-learn:", sklearn.__version__)
print("RDKit:", rdBase.rdkitVersion)

print("\nLabel meanings:")
print("  0 = Predicted Ames-negative")
print("  1 = Predicted Ames-positive (mutagenic)")

# ============================================================
# STEP 2: FINGERPRINT GENERATOR
# ============================================================

fingerprint_generator = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=FP_RADIUS,
        fpSize=FP_BITS
    )
)

def create_fingerprints(smiles_list):
    X = np.zeros(
        (len(smiles_list), FP_BITS),
        dtype=np.uint8
    )

    for i, smiles in enumerate(smiles_list):

        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            raise ValueError(
                f"Invalid molecule during Ames labelling: {smiles}"
            )

        fp = fingerprint_generator.GetFingerprint(mol)

        DataStructs.ConvertToNumpyArray(fp, X[i])

    return X


# ============================================================
# STEP 3: VALIDATE PREDICTIONS AND PROBABILITIES
# ============================================================

def predict_ames(smiles_list):

    X = create_fingerprints(smiles_list)

    predictions = np.asarray(rf.predict(X)).reshape(-1)
    probabilities = np.asarray(
        rf.predict_proba(X),
        dtype=np.float64
    )

    expected_shape = (len(smiles_list), len(classes))

    if probabilities.shape != expected_shape:
        raise ValueError(
            f"Invalid predict_proba shape: {probabilities.shape}"
        )

    if not np.isin(predictions, [0, 1]).all():
        raise ValueError("RF returned unknown class labels")

    if (
        not np.isfinite(probabilities).all()
        or (probabilities < -1e-8).any()
        or (probabilities > 1 + 1e-8).any()
        or not np.allclose(
            probabilities.sum(axis=1),
            1.0,
            atol=1e-5,
            rtol=1e-5
        )
    ):
        raise ValueError(
            "Invalid RF probabilities. Stop and audit "
            "the classifier/environment before labelling."
        )

    # Detect disagreement between predict and predict_proba
    probability_classes = np.asarray(classes)[
        probabilities.argmax(axis=1)
    ]

    if not np.array_equal(
        predictions,
        probability_classes
    ):
        raise ValueError(
            "RF predict() disagrees with predict_proba "
            "argmax. Verify whether a custom decision "
            "threshold is intended."
        )

    return (
        predictions.astype(int),
        probabilities[:, positive_idx]
    )


# ============================================================
# STEP 4: SMALL MODEL SMOKE TEST
# ============================================================

test_smiles = [
    "CCO",
    "CC(=O)O",
    "c1ccccc1",
    "CCN"
]

test_predictions, test_probabilities = predict_ames(
    test_smiles
)

print("\nRF smoke test passed")

for smi, pred, prob in zip(
    test_smiles,
    test_predictions,
    test_probabilities
):
    print(
        f"  {smi}: class={pred}, "
        f"P(Ames+)={prob:.4f}"
    )

# These are pipeline checks, not known experimental labels.

# ============================================================
# STEP 5: STREAM THE COMPLETE CHEMBL DATASET
# ============================================================

counts = Counter()

# Temporary files prevent incomplete outputs from appearing
# under the final filenames if processing fails.

TEMP_CSV = Path(str(OUTPUT_CSV) + ".partial")
TEMP_TXT = Path(str(OUTPUT_TXT) + ".partial")

with (
    INPUT_CSV.open(
        "r", encoding="utf-8", newline=""
    ) as infile,

    TEMP_CSV.open(
        "w", encoding="utf-8", newline=""
    ) as csvfile,

    TEMP_TXT.open(
        "w", encoding="utf-8"
    ) as txtfile,
):

    reader = csv.DictReader(infile)

    assert "canonical_smiles" in (reader.fieldnames or [])

    writer = csv.DictWriter(
        csvfile,
        fieldnames=[
            "chembl_id",
            "canonical_smiles",
            "ames_prediction",
            "ames_positive_probability"
        ]
    )

    writer.writeheader()

    while True:

        batch = list(islice(reader, BATCH_SIZE))

        if not batch:
            break

        smiles_batch = [
            row["canonical_smiles"].strip()
            for row in batch
        ]

        if any(not smi for smi in smiles_batch):
            raise ValueError("Empty SMILES in cleaned ChEMBL")

        predictions, probabilities = predict_ames(
            smiles_batch
        )

        for row, smiles, label, probability in zip(
            batch,
            smiles_batch,
            predictions,
            probabilities
        ):

            label = int(label)

            writer.writerow({
                "chembl_id": row.get("chembl_id", ""),
                "canonical_smiles": smiles,
                "ames_prediction": label,
                "ames_positive_probability": (
                    f"{float(probability):.6f}"
                )
            })

            # Required format for toxicity-aware pretraining:
            # SMILES followed by predicted Ames class.
            txtfile.write(f"{smiles} {label}\n")

            counts["total"] += 1
            counts[f"class_{label}"] += 1

        if counts["total"] % (BATCH_SIZE * 100) == 0:
            print(
                f"  Labelled {counts['total']:,} molecules"
            )


# ============================================================
# STEP 6: FINAL RECORD CHECK
# ============================================================

assert counts["total"] == EXPECTED_COUNT, (
    f"Expected {EXPECTED_COUNT:,} molecules, "
    f"got {counts['total']:,}"
)

assert (
    counts["class_0"] + counts["class_1"]
    == counts["total"]
)

# Promote complete temporary files to final filenames.
os.replace(TEMP_CSV, OUTPUT_CSV)
os.replace(TEMP_TXT, OUTPUT_TXT)

# ============================================================
# STEP 7: MODEL HASH AND REPORT
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


report = {
    "source_file": str(INPUT_CSV),
    "output_csv": str(OUTPUT_CSV),
    "output_txt": str(OUTPUT_TXT),
    "random_forest_file": str(RF_PATH),
    "random_forest_sha256": sha256_file(RF_PATH),
    "fingerprint": {
        "type": "Morgan bit vector",
        "radius": FP_RADIUS,
        "bits": FP_BITS
    },
    "label_meanings": {
        "0": "Predicted Ames-negative",
        "1": "Predicted Ames-positive (mutagenic)"
    },
    "total_molecules": counts["total"],
    "predicted_ames_negative": counts["class_0"],
    "predicted_ames_positive": counts["class_1"],
    "class_1_fraction": (
        counts["class_1"] / counts["total"]
    ),
    "sklearn_version": sklearn.__version__,
    "rdkit_version": rdBase.rdkitVersion,
    "labels_are_experimental": False,
    "rf_training_feature_configuration_verified": False
}

with REPORT_PATH.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("BLOCK 14 — AMES LABELLING SUMMARY")
print("=" * 65)

print(f"Total molecules:             {counts['total']:,}")
print(f"Predicted Ames-negative (0): {counts['class_0']:,}")
print(f"Predicted Ames-positive (1): {counts['class_1']:,}")

print(
    "Predicted Ames-positive rate:",
    f"{100 * counts['class_1'] / counts['total']:.2f}%"
)

print("\nSaved outputs:")
print(OUTPUT_CSV)
print(OUTPUT_TXT)
print(REPORT_PATH)

print("\nBLOCK 14 COMPLETED!")


BLOCK 14: RANDOM FOREST AMES LABELLING

Random Forest loaded successfully
Model: RandomForestClassifier
Classes: [0.0, 1.0]
Input features: 2048
scikit-learn: 1.3.2
RDKit: 2026.03.6

Label meanings:
  0 = Predicted Ames-negative
  1 = Predicted Ames-positive (mutagenic)

RF smoke test passed
  CCO: class=0, P(Ames+)=0.0200
  CC(=O)O: class=0, P(Ames+)=0.0200
  c1ccccc1: class=0, P(Ames+)=0.0800
  CCN: class=0, P(Ames+)=0.1500
  Labelled 51,200 molecules
  Labelled 102,400 molecules
  Labelled 153,600 molecules
  Labelled 204,800 molecules
  Labelled 256,000 molecules
  Labelled 307,200 molecules
  Labelled 358,400 molecules
  Labelled 409,600 molecules
  Labelled 460,800 molecules
  Labelled 512,000 molecules
  Labelled 563,200 molecules
  Labelled 614,400 molecules
  Labelled 665,600 molecules
  Labelled 716,800 molecules
  Labelled 768,000 molecules
  Labelled 819,200 molecules
  Labelled 870,400 molecules
  Labelled 921,600 molecules
  Labelled 972,800 molecules
  Labelled 1,024,000

## Final Dataset Integrity Verification

In [3]:

# ============================================================
# BLOCK 15: FINAL CTDDG PREPROCESSING INTEGRITY AUDIT
# ============================================================

import csv
import gzip
import json
import math
import hashlib
import os
from pathlib import Path
from itertools import zip_longest
from collections import Counter

ROOT = Path("/home/muhamed/repo/CTDDG/REWORKED_CODE")
OUT = ROOT / "PREPROCESSING_OUTPUT"

CHEMBL = OUT / "chembl"
BINDINGDB = OUT / "bindingdb"
VOCAB = OUT / "vocabulary"

CLEAN_CSV = CHEMBL / "chembl_clean.csv"
LABELLED_CSV = CHEMBL / "chembl_ames_predicted.csv"
FINAL_TXT = CHEMBL / "chembl_final.txt"

VOCAB_JSON = VOCAB / "atom_vocabulary.json"
VOCAB_TXT = VOCAB / "atom_types.txt"

AMES_REPORT = CHEMBL / "ames_labelling_report.json"
GRAPH_REPORT = (
    OUT / "graph_compatibility"
    / "graph_compatibility_report.json"
)
INTEGRITY_REPORT = BINDINGDB / "split_integrity_report.json"
MANIFEST = OUT / "preprocessing_manifest.json"

FINAL_REPORT = OUT / "preprocessing_final_audit.json"

EXPECTED_CHEMBL = 1_090_506
EXPECTED_BINDINGDB = 900_499
EXPECTED_ATOMS = 55
EXPECTED_BONDS = 4
EXPECTED_FOLDS = 10

print("=" * 65)
print("BLOCK 15: FINAL CTDDG PREPROCESSING AUDIT")
print("=" * 65)

# ============================================================
# 1. VERIFY REQUIRED OUTPUT FILES
# ============================================================

required = [
    CLEAN_CSV,
    LABELLED_CSV,
    FINAL_TXT,
    VOCAB_JSON,
    VOCAB_TXT,
    AMES_REPORT,
    GRAPH_REPORT,
    INTEGRITY_REPORT,
    MANIFEST,
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing output: {path}\n"
            "Make sure Block 14 completed successfully."
        )

# Reject incomplete Block 14 output
for path in [
    Path(str(LABELLED_CSV) + ".partial"),
    Path(str(FINAL_TXT) + ".partial"),
]:
    if path.exists():
        raise RuntimeError(
            f"Incomplete labelling file exists: {path}"
        )

print("\n[1/6] All required output files found: PASSED")

# ============================================================
# 2. VERIFY EVERY CHEMBL RECORD AND AMES LABEL
# ============================================================

counts = Counter()
probability_min = 1.0
probability_max = 0.0

print("\n[2/6] Verifying complete ChEMBL dataset...")

with (
    CLEAN_CSV.open(
        "r", encoding="utf-8", newline=""
    ) as clean_file,
    LABELLED_CSV.open(
        "r", encoding="utf-8", newline=""
    ) as labelled_file,
    FINAL_TXT.open(
        "r", encoding="utf-8"
    ) as txt_file,
):
    clean_reader = csv.DictReader(clean_file)
    labelled_reader = csv.DictReader(labelled_file)

    required_label_cols = {
        "canonical_smiles",
        "ames_prediction",
        "ames_positive_probability",
    }

    assert "canonical_smiles" in clean_reader.fieldnames
    assert required_label_cols.issubset(
        set(labelled_reader.fieldnames)
    )

    for row_number, triple in enumerate(
        zip_longest(
            clean_reader,
            labelled_reader,
            txt_file
        ),
        start=1
    ):
        clean_row, labelled_row, txt_line = triple

        if any(item is None for item in triple):
            raise ValueError(
                f"File lengths differ at row {row_number}"
            )

        clean_smiles = clean_row[
            "canonical_smiles"
        ].strip()

        labelled_smiles = labelled_row[
            "canonical_smiles"
        ].strip()

        txt_parts = txt_line.strip().rsplit(" ", 1)

        if len(txt_parts) != 2:
            raise ValueError(
                f"Invalid TXT format at row {row_number}"
            )

        txt_smiles, txt_label = txt_parts

        if not clean_smiles or not (
            clean_smiles == labelled_smiles == txt_smiles
        ):
            raise ValueError(
                f"SMILES mismatch at row {row_number}"
            )

        label_str = labelled_row[
            "ames_prediction"
        ].strip()

        if label_str not in {"0", "1"}:
            raise ValueError(
                f"Invalid Ames label at row {row_number}"
            )

        if txt_label != label_str:
            raise ValueError(
                f"TXT/CSV label mismatch at row {row_number}"
            )

        probability = float(
            labelled_row["ames_positive_probability"]
        )

        if not (
            math.isfinite(probability)
            and 0 <= probability <= 1
        ):
            raise ValueError(
                f"Invalid probability at row {row_number}"
            )

        probability_min = min(
            probability_min, probability
        )
        probability_max = max(
            probability_max, probability
        )

        counts["total"] += 1
        counts[f"class_{label_str}"] += 1

        if counts["total"] % 200_000 == 0:
            print(
                f"  Verified {counts['total']:,} molecules"
            )

assert counts["total"] == EXPECTED_CHEMBL

print("\nChEMBL verification: PASSED")
print("Total molecules:", f"{counts['total']:,}")
print("Ames-negative (0):", f"{counts['class_0']:,}")
print("Ames-positive (1):", f"{counts['class_1']:,}")

# ============================================================
# 3. VERIFY AMES LABELLING REPORT
# ============================================================

print("\n[3/6] Verifying Ames labelling metadata...")

with AMES_REPORT.open("r", encoding="utf-8") as f:
    ames = json.load(f)

assert ames["total_molecules"] == counts["total"]

assert (
    ames["predicted_ames_negative"]
    == counts["class_0"]
)

assert (
    ames["predicted_ames_positive"]
    == counts["class_1"]
)

assert ames["fingerprint"]["radius"] == 2
assert ames["fingerprint"]["bits"] == 2048
assert ames["labels_are_experimental"] is False

# Confirm model file still matches the labelling report
rf_path = Path(ames["random_forest_file"])

assert rf_path.is_file(), "Original RF model is missing"

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

assert (
    sha256_file(rf_path)
    == ames["random_forest_sha256"]
), "RF model has changed since labelling"

print("Ames metadata and RF checksum: PASSED")

# ============================================================
# 4. VERIFY ATOM VOCABULARY
# ============================================================

print("\n[4/6] Verifying atom vocabulary...")

with VOCAB_JSON.open("r", encoding="utf-8") as f:
    vocabulary = json.load(f)

with MANIFEST.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

atom_entries = vocabulary["atom_types"]

assert len(atom_entries) == EXPECTED_ATOMS

assert [
    entry["id"] for entry in atom_entries
] == list(range(EXPECTED_ATOMS))

expected_types = [
    (
        entry["symbol"],
        int(entry["formal_charge"]),
        int(entry["explicit_hydrogens"])
    )
    for entry in atom_entries
]

with VOCAB_TXT.open("r", encoding="utf-8") as f:
    actual_types = [
        (
            parts[0],
            int(parts[1]),
            int(parts[2])
        )
        for line in f if line.strip()
        for parts in [line.strip().split(",")]
    ]

assert actual_types == expected_types
assert len(set(actual_types)) == EXPECTED_ATOMS

vocab_hash = sha256_file(VOCAB_TXT)

assert (
    vocab_hash == manifest["vocabulary"]["sha256"]
), "Atom vocabulary changed after Block 13"

assert (
    manifest["vocabulary"]["num_bond_types"]
    == EXPECTED_BONDS
)

print("Atom vocabulary: PASSED")
print("Atom types:", len(actual_types))
print("Bond types:", EXPECTED_BONDS)
print("Vocabulary SHA256:", vocab_hash)

# ============================================================
# 5. VERIFY BINDINGDB FOLDS AND PREVIOUS AUDITS
# ============================================================

print("\n[5/6] Verifying BindingDB outputs...")

with INTEGRITY_REPORT.open(
    "r", encoding="utf-8"
) as f:
    integrity = json.load(f)

assert integrity["raw_pair_alignment_check"] == "passed"
assert integrity["exact_protein_overlap_check"] == "passed"

fold_files = sorted(
    BINDINGDB.rglob("pairs_clean.csv.gz")
)

assert len(fold_files) == EXPECTED_FOLDS, (
    f"Expected 10 folds, found {len(fold_files)}"
)

fold_counts = {}
bindingdb_total = 0

for file_path in fold_files:

    with gzip.open(
        file_path, "rt",
        encoding="utf-8",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        assert {
            "source_line",
            "protein_sequence",
            "canonical_smiles"
        }.issubset(set(reader.fieldnames))

        num_rows = 0

        for row in reader:
            if not (
                row["source_line"].strip()
                and row["protein_sequence"].strip()
                and row["canonical_smiles"].strip()
            ):
                raise ValueError(
                    f"Empty BindingDB field in {file_path}"
                )

            num_rows += 1

    name = f"{file_path.parent.parent.name}/{file_path.parent.name}"

    fold_counts[name] = num_rows
    bindingdb_total += num_rows

    print(f"  {name}: {num_rows:,}")

assert bindingdb_total == EXPECTED_BINDINGDB

assert (
    bindingdb_total == integrity["total_retained_pairs"]
)

assert (
    integrity["total_raw_pairs"]
    - integrity["total_filtered_pairs"]
    == bindingdb_total
)

print("\nBindingDB verification: PASSED")
print("Total retained pairs:", f"{bindingdb_total:,}")

with GRAPH_REPORT.open("r", encoding="utf-8") as f:
    graph_report = json.load(f)

sample_graphs = graph_report["sampled_molecules"]

assert sample_graphs["failed"] == 0
assert sample_graphs["passed"] > 0

print("\nPrevious graph encoding sample: PASSED")
print("Sample graphs checked:", sample_graphs["passed"])

# ============================================================
# 6. SAVE FINAL AUDIT AND READINESS SUMMARY
# ============================================================

print("\n[6/6] Preparing final audit report...")

# This does not claim that all molecular graphs were
# tested, or that protein similarity has been exhaustively
# verified. Those are separate checks.

report = {
    "status": "PREPROCESSING_FILE_INTEGRITY_PASSED",

    "chembl": {
        "clean_molecules": counts["total"],
        "labelled_molecules": counts["total"],
        "predicted_ames_negative": counts["class_0"],
        "predicted_ames_positive": counts["class_1"],
        "ames_positive_fraction": (
            counts["class_1"] / counts["total"]
        ),
        "probability_min": probability_min,
        "probability_max": probability_max,
        "csv_txt_alignment_verified": True,
        "labels_are_experimental": False,
        "labelled_csv": str(LABELLED_CSV),
        "pretraining_txt": str(FINAL_TXT),
    },

    "bindingdb": {
        "retained_pairs": bindingdb_total,
        "folds": fold_counts,
        "raw_pair_alignment_verified_by_block_10": True,
        "exact_protein_overlap_checked_by_block_10": True,
        "exhaustive_protein_similarity_verified": False,
    },

    "molecular_graphs": {
        "num_atom_types": EXPECTED_ATOMS,
        "num_bond_types": EXPECTED_BONDS,
        "vocabulary_sha256": vocab_hash,
        "sample_graphs_passed": sample_graphs["passed"],
        "sample_graphs_failed": sample_graphs["failed"],
        "full_graph_encoding_verified": False,
        "stereochemistry_explicitly_encoded": False,
    },

    "ames_classifier": {
        "model_file": str(rf_path),
        "sha256": ames["random_forest_sha256"],
        "fingerprint_radius": 2,
        "fingerprint_bits": 2048,
        "training_feature_configuration_verified": ames.get(
            "rf_training_feature_configuration_verified",
            False
        ),
    },

    "next_stage": {
        "name": "CTDDG_PRETRAINING",
        "from_scratch": True,
        "n_atom": EXPECTED_ATOMS,
        "n_bond": EXPECTED_BONDS,
        "atom_types_file": str(VOCAB_TXT),
        "chembl_file": str(FINAL_TXT),
    }
}

temporary_report = Path(str(FINAL_REPORT) + ".partial")

with temporary_report.open(
    "w", encoding="utf-8"
) as f:
    json.dump(report, f, indent=2)

os.replace(temporary_report, FINAL_REPORT)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FINAL CTDDG PREPROCESSING SUMMARY")
print("=" * 65)

print(f"ChEMBL molecules:     {counts['total']:,}")
print(f"Predicted Ames - (0): {counts['class_0']:,}")
print(f"Predicted Ames + (1): {counts['class_1']:,}")
print(f"BindingDB pairs:      {bindingdb_total:,}")
print(f"BindingDB fold files: {len(fold_files)}")
print(f"Atom types:           {EXPECTED_ATOMS}")
print(f"Bond types:           {EXPECTED_BONDS}")

print("\nFile consistency checks: PASSED")
print("Vocabulary consistency:   PASSED")
print("Previous alignment audit: PASSED")

print("\nFinal report:")
print(FINAL_REPORT)

print("\nNEXT STAGE: CTDDG PRETRAINING FROM SCRATCH")
print("n_atom =", EXPECTED_ATOMS)
print("n_bond =", EXPECTED_BONDS)

print("\nBLOCK 15 COMPLETED!")


BLOCK 15: FINAL CTDDG PREPROCESSING AUDIT

[1/6] All required output files found: PASSED

[2/6] Verifying complete ChEMBL dataset...
  Verified 200,000 molecules
  Verified 400,000 molecules
  Verified 600,000 molecules
  Verified 800,000 molecules
  Verified 1,000,000 molecules

ChEMBL verification: PASSED
Total molecules: 1,090,506
Ames-negative (0): 781,280
Ames-positive (1): 309,226

[3/6] Verifying Ames labelling metadata...
Ames metadata and RF checksum: PASSED

[4/6] Verifying atom vocabulary...
Atom vocabulary: PASSED
Atom types: 55
Bond types: 4
Vocabulary SHA256: 3f2a10321e1301f7a45f6ebb260181c70c0e28ae5c3162bd5c3e713db335184f

[5/6] Verifying BindingDB outputs...
  test/test_4_org_1000_112: 16,200
  test/test_4_org_1002_103: 16,015
  test/test_4_org_1004_122: 26,640
  test/test_4_org_1036_124: 25,875
  test/test_4_org_1042_104: 11,049
  train/train_4_org_1000_112: 143,999
  train/train_4_org_1002_103: 144,884
  train/train_4_org_1004_122: 146,059
  train/train_4_org_1036_124

KeyError: 'failed'